# Collection Équinoxe — hausse de loyer 2026

**Réponse : +3,2 %** de hausse du **loyer réellement encaissé** (`sRentEffective`) d'un **même logement** (`sPropCode + sUnitCode`) par rapport à son bail précédent, pour les baux débutant en 2026 (moyenne des hausses médianes des 4 segments, pondérée par les baux attendus en 2026). Hausse au bail : +5,8 %.

- **Comparaison avec 2025, sur la même mesure** : 4,0 % encaissé, 7,4 % au bail (la médiane de toutes les paires, section 4, donne 3,6 % et 7,0 %).
- **Scénarios selon les concessions** : 2,4 % (elles montent comme en 2025) / **3,2 %** (rythme moyen 2024–2025) / 4,1 % (montée divisée par 2) / 5,8 % (leur part cesse de monter).
- **Erreur moyenne historique** (MAE, 2023–2025) : 1,6 pt ; pire année : 2,4 pt. Ce n'est pas un intervalle de confiance.
- **Taux du TAL 2026** (3,1 %) : publié en janvier 2026, après le 31 décembre mais avant les avis de renouvellement ; s'il était inconnu : 4,4 %.

**Plan** (sections = tâches des consignes)

| Section | Contenu |
|---|---|
| 1 | Définir la question |
| 2 | Données : chargement, colonnes, renommage, concessions rattachées aux baux, effet de composition |
| 3 | Croissance à unité constante (`sPropCode + sUnitCode`) |
| 4 | Concessions : loyer au bail et loyer encaissé |
| 5 | Renouvellements, relocations, règles du Québec et de l'Ontario |
| 6 | Données publiques et réconciliation |
| 7 | Prévision et backtest : modèles, `estimate_2026()`, `backtest()`, scénarios, hypothèses |
| 8 | Bonus : prévision par immeuble et par nombre de chambres |
| Annexes | Explicabilité, modèle à deux niveaux, tableau de bord |

Les graphiques ont des étiquettes en anglais : ceux qui ont un nom sont enregistrés dans `figures/` et repris tels quels dans la présentation.

## 1. Définir la question

**Hausse mesurée : le loyer encaissé (`sRentEffective`) d'un même logement par rapport à son bail précédent, pour les baux débutant en 2026.**

- **Encaissé plutôt qu'au bail** : un budget de revenus se fait avec l'argent réellement perçu. Les concessions (mois gratuits, stationnement, casier) représentent maintenant ~9 % du loyer et grossissent : la hausse au bail surestime ce que le propriétaire touche (section 4). On montre aussi la hausse au bail.
- **Même logement** : la médiane de tous les baux mesure surtout quels immeubles louent (section 2).
- **Par segment** : renouvellements et nouveaux locataires, Québec et Ontario suivent des règles différentes (section 5) ; le chiffre du portefeuille est la moyenne des 4 segments pondérée par les baux attendus en 2026.
- **Année = début du bail** : c'est là que le nouveau loyer entre dans les revenus.

## 2. Données

### 2.1 Chargement

Les quatre tables du CRM (jamais modifiées) et les sources publiques, téléchargées et nettoyées par `public_data.py` (une fonction par source).

| Source publique | Ce qu'on en tire | Section |
|---|---|---|
| IPC, Statistique Canada (18-10-0004-01) | hausse des loyers payés au Québec et en Ontario, avec date de publication | 6, 7 |
| TAL et ligne directrice ontarienne | repère des renouvellements ; exemption de The Met | 5, 7 |
| SCHL, Enquête sur les logements locatifs | hausse à échantillon constant, logements avec / sans rotation, panel des villes | 6, 7 |
| FPI concurrentes (InterRent, Killam, CAPREIT, Minto) | hausses de renouvellement et de relocation des concurrents | 6 |
| Kaggle, 25 000 annonces (juin 2024) | prix du marché autour de chaque immeuble | 6 |

Toutes les constantes sont définies ici.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import public_data

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Années de l'analyse : une seule source pour tout le notebook
FORECAST_YEAR = 2026                     # année à prévoir
LAST_OBSERVED_YEAR = FORECAST_YEAR - 1
BACKTEST_YEARS = [2023, 2024, 2025]      # années de validation demandées par le défi
FIRST_TEST_YEAR = 2022                   # première année prévisible avec deux ans d'historique
FIRST_INDEX_YEAR = 2018                  # première année d'indice
SCORING_START = 2021                     # première année notée (historique suffisant)
DISPLAY_START_YEAR = 2020                # début des graphiques
ANALYSIS_YEARS = range(2019, FORECAST_YEAR)
DISPLAY_YEARS = list(range(DISPLAY_START_YEAR, FORECAST_YEAR))
ONTARIO_EXEMPTION_DATE = "2018-11-15"    # première occupation après cette date : exempté (ontario.ca)
UNIT_KEY = ["sPropCode", "sUnitCode"]    # jamais sSite + sUnitCode

# Graphiques : charte JADCO, étiquettes en anglais ; ceux qui ont un nom sont repris tels quels dans la présentation
CHART_NAVY, CHART_GOLD, CHART_SAND, CHART_GREY, CHART_LIGHT = "#0B1F40", "#8B7543", "#B9AB88", "#8A93A0", "#E3E6EC"
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
FIG_SIZE, FIG_DPI = (10.4, 5.2), 160
plt.rcParams.update({"font.size": 14, "axes.edgecolor": CHART_GREY, "axes.labelcolor": CHART_NAVY, "xtick.color": CHART_NAVY,
                     "ytick.color": CHART_NAVY, "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})

def finish(fig, name=None):
    """Affiche la figure ; avec `name`, l'enregistre aussi dans figures/ pour la présentation."""
    fig.tight_layout()
    if name:
        fig.savefig(FIG_DIR / f"{name}.png", dpi=FIG_DPI)
    plt.show()

def pct_axis(ax, axis="y"):
    """Axe en %, quadrillage léger."""
    getattr(ax, f"{axis}axis").set_major_formatter(lambda v, _: f"{v:.0f}%")
    ax.grid(axis=axis, color=CHART_LIGHT, lw=0.8)

# Données CRM : fichiers sources jamais modifiés
leases_raw = pd.read_csv("equinoxe_lease_history.csv", parse_dates=["sSignDate", "sLeaseFrom", "sLeaseTo", "sAvailable"])
concessions_raw = pd.read_csv("equinoxe_concessions.csv", parse_dates=["sDateFrom", "sDateTo"])
asking_raw = pd.read_csv("equinoxe_asking_history.csv")
listings_raw = pd.read_csv("equinoxe_listings.csv")

# Données publiques déjà nettoyées par public_data.py (True : tout retélécharger, internet requis)
REBUILD_PUBLIC_DATA = False
if REBUILD_PUBLIC_DATA:
    public_data.build_all()
EXTERNAL_DIR = Path("external/clean")
EXTERNAL_FILES = {
    "cpi_annual": "statcan_cpi_annual.csv", "cpi_monthly": "statcan_cpi_monthly.csv",
    "regulation": "rent_regulation_annual.csv", "cmhc_zone": "cmhc_zone_panel.csv",
    "cmhc_centre": "cmhc_centre_panel.csv", "cmhc_turnover": "cmhc_turnover_vs_sitting.csv",
    "cmhc_building_map": "cmhc_building_zone_map.csv", "reits": "reit_metrics.csv",
}
DATE_COLUMNS = {"cpi_monthly": ["month", "released_at"]}
external = {name: pd.read_csv(EXTERNAL_DIR / f, parse_dates=DATE_COLUMNS.get(name, False)) for name, f in EXTERNAL_FILES.items()}

for name, df in [("baux", leases_raw), ("concessions", concessions_raw), ("loyers affichés", asking_raw), ("logements", listings_raw)]:
    print(f"{name:<16} {len(df):>6,} lignes  {df.shape[1]:>2} colonnes")
print("sources publiques :", ", ".join(external))

### 2.2 Ce que chaque colonne contient vraiment

Convention Yardi : les colonnes **`h`** (*handle*) sont des identifiants pour les jointures, les colonnes **`s`** (*stored value*) servent à l'affichage et au regroupement ; `s` ne veut pas dire « texte » (`sRent` est un nombre). Plusieurs descriptions ne correspondent pas aux données :
| Table | Colonne | Constat | Décision |
|---|---|---|---|
| baux | `sAvailable` | toujours égale à `sLeaseFrom` | supprimée |
| baux | `sLeaseTerm` | une **étiquette d'annonce**, pas la durée : « 6 Months » sur 100 baux de 12 mois | renommée `listing_term_label` ; la vraie durée est `sTermMonths` |
| baux | `sRentEffective` | `sRent` + concessions ÷ durée ; inclut stationnement, casier | `rent_effective_crm` |
| concessions | `sAmount` | montant **par mois**, à multiplier par `sMonths` | `amount_per_month` |
| concessions | `PromoPay` | décrit comme mensuel, c'est un rabais **ponctuel** (≈ 0,8 mois de loyer) | étalé sur le bail |
| loyers affichés | `sAskingRent` | observé seulement autour de la signature ; loyer signé ≈ 2 % sous l'affiché | signal testé, rejeté (7.2) |
| logements | `sRent` | loyer **affiché actuel** | `asking_rent_current` |

In [ ]:
DAYS_PER_MONTH = 365.25 / 12
checks = {
    "hUnit correspond 1 pour 1 à (immeuble, logement)": leases_raw.groupby("hUnit")[UNIT_KEY].nunique().max().max() == 1
                                             and leases_raw.groupby(UNIT_KEY).hUnit.nunique().max() == 1,
    "aucun doublon (logement, début de bail)": not leases_raw.duplicated(UNIT_KEY + ["sLeaseFrom"]).any(),
    "sAvailable == sLeaseFrom": (leases_raw.sAvailable == leases_raw.sLeaseFrom).all(),
    "sTermMonths cohérent avec les dates (< 0,1 mois)": ((leases_raw.sLeaseTo - leases_raw.sLeaseFrom).dt.days / DAYS_PER_MONTH
                                                - leases_raw.sTermMonths).abs().max() < 0.1,
    "premier bail de chaque logement = seq 1, non renouvelé": leases_raw[leases_raw.sTermSeq == 1].sRenewal.eq(0).all(),
    "caractéristiques du logement constantes d'un bail à l'autre": all(leases_raw.groupby(UNIT_KEY)[c].nunique().max() == 1
                                                  for c in ["sBeds", "sBaths", "sSqft", "sFloor", "sUnitType"]),
    "sLeaseTerm varie pour un même logement (étiquette, pas durée)": leases_raw.groupby(UNIT_KEY).sLeaseTerm.nunique().max() > 1,
}
for name, ok in checks.items():
    print(f"{'OUI' if ok else 'NON'} {name}")
print()
print("étiquette sLeaseTerm (lignes) contre durée réelle en mois (colonnes) :")
print(pd.crosstab(leases_raw.sLeaseTerm.replace({"6 Months": "Six months"}), leases_raw.sTermMonths.round(), margins=True))

### 2.3 Tables renommées

Les `h` (identifiants) servent aux jointures, les `s` (valeurs) à l'affichage. On ajoute `province_regime` (`QC_TAL` pour les cinq immeubles québécois, `ON_GUIDELINE` pour The Met, c'est-à-dire le régime ontarien : The Met en est exempté, voir 11) et `lease_year` (année de début du bail).

In [ ]:
LEASE_COLUMNS = {
    "hProperty": "property_id", "hUnit": "unit_id", "hBuilding": "building_id",
    "sPropCode": "prop_code", "sUnitCode": "unit_code", "sBuilding": "building", "sSite": "site",
    "sCity": "city", "sState": "province", "sAddr1": "address", "sLatitude": "latitude", "sLongitude": "longitude",
    "sPropType": "property_type", "sUnitType": "unit_layout", "sUnitSubtype": "bed_label",
    "sBeds": "beds", "sBaths": "baths", "sSqft": "sqft", "sFloor": "floor", "sFurnishing": "furnishing",
    "sSmoking": "smoking_policy", "sCats": "cats_allowed", "sDogs": "dogs_allowed", "sLink": "unit_link",
    "sLeaseTerm": "listing_term_label", "sSignDate": "sign_date", "sLeaseFrom": "lease_start", "sLeaseTo": "lease_end",
    "sTermMonths": "term_months", "sTermSeq": "lease_seq", "sRenewal": "is_renewal",
    "sRent": "rent_contract", "sRentEffective": "rent_effective_crm", "sConcession": "has_concession",
}
REGIME_BY_PROVINCE = {"Quebec": "QC_TAL", "Ontario": "ON_GUIDELINE"}

lease_df = (leases_raw.drop(columns=["sAvailable"]).rename(columns=LEASE_COLUMNS)
            .sort_values(["prop_code", "unit_code", "lease_start"]).reset_index(drop=True))
lease_df["lease_id"] = np.arange(len(lease_df))
lease_df["is_renewal"] = lease_df.is_renewal.astype(bool)
lease_df["has_concession"] = lease_df.has_concession.astype(bool)
lease_df["lease_year"] = lease_df.lease_start.dt.year
lease_df["province_regime"] = lease_df.province.map(REGIME_BY_PROVINCE)
assert lease_df.province_regime.notna().all()

CONCESSION_COLUMNS = {"hUnit": "unit_id", "sPropCode": "prop_code", "sUnitCode": "unit_code", "sBuilding": "building",
                      "sChargeCode": "charge_code", "sChargeDesc": "charge_desc", "sAmount": "amount_per_month",
                      "sDateFrom": "concession_start", "sDateTo": "concession_end", "sMonths": "months"}
concession_df = concessions_raw.rename(columns=CONCESSION_COLUMNS)[list(CONCESSION_COLUMNS.values())].copy()
concession_df["concession_id"] = np.arange(len(concession_df))

asking_df = asking_raw.rename(columns={"hUnit": "unit_id", "sPropCode": "prop_code", "sUnitCode": "unit_code",
                                       "sBuilding": "building", "sAskingRent": "asking_rent", "sMonth": "month"})
asking_df["month"] = pd.to_datetime(asking_df.month + "-01")

unit_df = listings_raw.rename(columns={**LEASE_COLUMNS, "sRent": "asking_rent_current", "sAsOf": "asking_as_of",
                                       "sUnitDesc": "unit_description", "sAvailable": "listing_available"})
print(lease_df.shape, concession_df.shape, asking_df.shape, unit_df.shape)
lease_df.head(3)

### 2.4 Concessions rattachées aux baux

Chaque concession est rattachée au bail du même logement qui commence le même mois ou avant. Valeur = `amount_per_month × months` (négative = rabais). Trois groupes : **rabais de loyer** (`freerent`, `PromoPay`), **accessoires** (`freepark`, `freelock`, `freeappl`), **autres** (`Freeothe`, `Indem`, `Referenc`).

In [ ]:
CONCESSION_GROUP = {"freerent": "rent_discount", "PromoPay": "rent_discount",
                    "freepark": "ancillary", "freelock": "ancillary", "freeappl": "ancillary",
                    "Freeothe": "other", "Indem": "other", "Referenc": "other"}
concession_df["concession_group"] = concession_df.charge_code.map(CONCESSION_GROUP)
assert concession_df.concession_group.notna().all()
concession_df["concession_value"] = concession_df.amount_per_month * concession_df.months.round()

lease_start_month = lease_df.lease_start.dt.to_period("M").dt.to_timestamp()
candidates = concession_df.merge(
    lease_df[["prop_code", "unit_code", "lease_id"]].assign(start_month=lease_start_month),
    on=["prop_code", "unit_code"])
linked = (candidates[candidates.concession_start >= candidates.start_month]
          .sort_values("start_month").drop_duplicates("concession_id", keep="last"))
concession_df = concession_df.merge(linked[["concession_id", "lease_id"]], on="concession_id", how="left")

print(f"concessions rattachées : {concession_df.lease_id.notna().sum():,} / {len(concession_df):,}")
leases_with_link = set(concession_df.lease_id.dropna().astype(int))
print("baux rattachés == baux marqués has_concession :",
      leases_with_link == set(lease_df.loc[lease_df.has_concession, "lease_id"]))

promo = concession_df[concession_df.charge_code == "PromoPay"].merge(lease_df[["lease_id", "rent_contract"]], on="lease_id")
print("\nPromoPay, nombre de mois :", promo.months.round().value_counts().to_dict())
print("PromoPay / loyer mensuel au bail :", (-promo.amount_per_month / promo.rent_contract).describe().round(2).to_dict())

On reconstruit `sRentEffective` = loyer + concessions ÷ durée : exact au dollar près pour ~91 % des baux. Les autres sont surtout des `PromoPay` d'un mois complet ; on garde le chiffre du CRM comme référence. Variante **loyer seulement** : `sRentEffective` sans les avantages accessoires et autres crédits.

In [ ]:
RECONCILIATION_TOLERANCE = 1.0      # dollars per month

value_by_group = (concession_df.pivot_table(index="lease_id", columns="concession_group",
                                            values="concession_value", aggfunc="sum")
                  .reindex(columns=["rent_discount", "ancillary", "other"]).fillna(0))
lease_df = lease_df.drop(columns=[c for c in value_by_group.columns if c in lease_df], errors="ignore")
lease_df = lease_df.join(value_by_group.add_prefix("concession_"), on="lease_id")
group_cols = ["concession_rent_discount", "concession_ancillary", "concession_other"]
lease_df[group_cols] = lease_df[group_cols].fillna(0)
lease_df["concession_total"] = lease_df[group_cols].sum(axis=1)

lease_df["rent_effective_rebuilt"] = lease_df.rent_contract + lease_df.concession_total / lease_df.term_months
lease_df["effective_matches_concessions"] = ((lease_df.rent_effective_rebuilt - lease_df.rent_effective_crm).abs()
                                             < RECONCILIATION_TOLERANCE)
lease_df["rent_effective_rent_only"] = (lease_df.rent_effective_crm
                                        - (lease_df.concession_ancillary + lease_df.concession_other) / lease_df.term_months)

match_rate = lease_df.effective_matches_concessions.mean()
print(f"sRentEffective reconstruit à {RECONCILIATION_TOLERANCE:.0f} $/mois près pour {match_rate:.1%} des baux")
mismatch = lease_df[~lease_df.effective_matches_concessions]
codes_of = concession_df.groupby("lease_id").charge_code.apply(lambda s: "+".join(sorted(s)))
print("\nconcessions des baux non reconstruits :")
print(mismatch.lease_id.map(codes_of).value_counts().head(6))

### 2.5 Effet de composition

La médiane de tous les baux donne +10,8 % en 2023, une hausse qui n'a pas eu lieu. Le Carlyle et The Met (2023), puis Westpark (2024), commencent à louer, à des loyers au pi² bien plus élevés : la médiane annuelle mesure surtout ce changement de composition.

In [ ]:
naive_median = lease_df[lease_df.lease_year.isin(ANALYSIS_YEARS)].groupby("lease_year").rent_contract.median()
print("Variation de la médiane de tous les baux (%) :", (naive_median.pct_change() * 100).round(1).dropna().to_dict())
mix_counts = lease_df[lease_df.lease_year.isin(ANALYSIS_YEARS)].pivot_table(
    index="lease_year", columns="building", values="lease_id", aggfunc="size").fillna(0)
psf_by_building = (lease_df.assign(rent_per_sqft=lease_df.rent_contract / lease_df.sqft)
                   .query("lease_year in @ANALYSIS_YEARS")
                   .pivot_table(index="lease_year", columns="building", values="rent_per_sqft", aggfunc="median"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
(mix_counts.div(mix_counts.sum(axis=1), axis=0) * 100).plot.bar(stacked=True, ax=axes[0], legend=False, colormap="cividis")
axes[0].set_ylabel("share of leases started (%)"); axes[0].set_xlabel(""); axes[0].set_title("Leases by building")
psf_by_building.plot(marker="o", ax=axes[1], colormap="cividis")
axes[1].set_ylabel("median contract rent per sq ft ($)"); axes[1].set_xlabel(""); axes[1].set_title("Price level by building")
axes[1].legend(fontsize=9, ncol=2); axes[1].grid(alpha=0.3)
finish(fig)

## 3. Croissance à unité constante

### 3.1 Même logement, bail contre bail précédent

Chaque bail est comparé au bail précédent du même logement (clé `sPropCode + sUnitCode`), et la hausse est annualisée. On exclut les paires après un bail de 6 mois (`MIN_GAP_YEARS`) et les écarts de plus de 2,5 ans (`MAX_GAP_YEARS`). On calcule la médiane et la moyenne pondérée par le loyer précédent, pour trois loyers : au bail, encaissé, et encaissé hors avantages accessoires.

In [ ]:
MIN_GAP_YEARS, MAX_GAP_YEARS = 0.75, 2.5
DAYS_PER_YEAR = 365.25
RENT_MEASURES = ["rent_contract", "rent_effective_crm", "rent_effective_rent_only"]

def build_pairs(leases, measures=RENT_MEASURES):
    # One row per lease that has a previous lease on the same unit.
    d = leases.sort_values(["prop_code", "unit_code", "lease_start"]).copy()
    by_unit = d.groupby(["prop_code", "unit_code"])
    d["prev_lease_start"] = by_unit.lease_start.shift(1)
    for m in measures:
        d[f"prev_{m}"] = by_unit[m].shift(1)
    d = d.dropna(subset=["prev_lease_start"]).copy()
    d["gap_years"] = (d.lease_start - d.prev_lease_start).dt.days / DAYS_PER_YEAR
    for m in measures:
        d[f"growth_{m}"] = ((d[m] / d[f"prev_{m}"]) ** (1 / d.gap_years) - 1) * 100
    return d

all_pairs = build_pairs(lease_df)
short_gap = all_pairs[all_pairs.gap_years < MIN_GAP_YEARS]
pairs = all_pairs[all_pairs.gap_years.between(MIN_GAP_YEARS, MAX_GAP_YEARS)].copy()
print(f"{len(all_pairs):,} paires consécutives ; {len(pairs):,} gardées ; "
      f"{len(short_gap)} exclues après un bail de 6 mois (hausse annualisée médiane : "
      f"{short_gap.growth_rent_contract.median():.1f}%)")

def yearly_growth(df, measure, by=None, years=ANALYSIS_YEARS):
    keys = ["lease_year"] + ([by] if isinstance(by, str) else list(by or []))
    g = df[df.lease_year.isin(years)].groupby(keys)
    out = pd.DataFrame({
        "pairs": g.size(),
        "median": g[f"growth_{measure}"].median(),
        "weighted_mean": g.apply(lambda x: np.average(x[f"growth_{measure}"], weights=x["prev_rent_contract"]),
                                 include_groups=False),
    })
    return out.round(2)

headline = pd.concat({m: yearly_growth(pairs, m)[["median", "weighted_mean"]] for m in RENT_MEASURES}, axis=1)
headline.insert(0, "pairs", yearly_growth(pairs, "rent_contract")["pairs"])

naive_growth = lease_df.groupby("lease_year").rent_contract.median().pct_change() * 100
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.plot(DISPLAY_YEARS, naive_growth.loc[DISPLAY_YEARS], "--o", color=CHART_GREY, lw=3, ms=9, label="Median of all leases")
ax.plot(DISPLAY_YEARS, headline.loc[DISPLAY_YEARS, ("rent_contract", "median")], "-o", color=CHART_GOLD, lw=4, ms=10, label="Same unit")
pct_axis(ax); ax.legend(loc="upper left"); finish(fig, "fig_mix")

### 3.2 Vérification de la clé

Les paires construites sur `hUnit` doivent être identiques à celles sur `sPropCode + sUnitCode` ; on compte aussi les comparaisons fausses qu'on ferait avec `sSite + sUnitCode`.

In [ ]:
by_handle = build_pairs(lease_df.assign(prop_code="hUnit", unit_code=lease_df.unit_id.astype(str)))
same_pairs = (len(by_handle) == len(all_pairs)
              and np.allclose(np.sort(by_handle.growth_rent_contract.values), np.sort(all_pairs.growth_rent_contract.values)))
print(f"paires sur hUnit identiques aux paires sur sPropCode + sUnitCode : {same_pairs}")

by_site = lease_df.assign(prop_code=lease_df.site).sort_values(["prop_code", "unit_code", "lease_start"])
previous_unit = by_site.groupby(["prop_code", "unit_code"]).unit_id.shift(1)
wrong = (previous_unit.notna() & (previous_unit != by_site.unit_id)).sum()
print(f"avec sSite + sUnitCode, {wrong:,} des {previous_unit.notna().sum():,} comparaisons « bail précédent » "
      f"compareraient deux logements DIFFÉRENTS")

### 3.3 Trois méthodes indépendantes

Indice de ventes répétées (effet fixe par logement + effet par année), indice hédonique (logements semblables), paires consécutives : elles s'accordent, la médiane naïve non.

Indice hédonique : logements **semblables** (immeuble, chambres, salles de bain, superficie, étage) plutôt qu'identiques.

In [ ]:
INDEX_BASE_YEAR = 2023        # first year all three new buildings have leases; base = 100

def repeat_rent_index(leases, measure, years=range(FIRST_INDEX_YEAR, FORECAST_YEAR)):
    d = leases[leases.lease_year.isin(years)].copy()
    d["log_rent"] = np.log(d[measure])
    year_dummies = pd.get_dummies(d.lease_year, dtype=float)
    unit_key = d.prop_code + "|" + d.unit_code
    y = d.log_rent - d.groupby(unit_key).log_rent.transform("mean")
    X = year_dummies - year_dummies.groupby(unit_key).transform("mean")
    X = X.drop(columns=INDEX_BASE_YEAR)                     # base year = 0
    coef, *_ = np.linalg.lstsq(X.values, y.values, rcond=None)
    effects = pd.Series(coef, index=X.columns)
    effects.loc[INDEX_BASE_YEAR] = 0.0
    return (np.exp(effects.sort_index()) * 100).rename(measure)

index_table = {}
for regime, sub in lease_df.groupby("province_regime"):
    for measure in ["rent_contract", "rent_effective_crm"]:
        index_table[(regime, measure)] = repeat_rent_index(sub, measure)
index_table = pd.DataFrame(index_table)
index_growth = (index_table.pct_change() * 100).round(2)

In [ ]:
def hedonic_index(leases, measure, years=range(FIRST_INDEX_YEAR, FORECAST_YEAR)):
    d = leases[leases.lease_year.isin(years)].copy()
    d["log_rent"] = np.log(d[measure]); d["log_sqft"] = np.log(d.sqft)
    model = smf.ols(f"log_rent ~ C(prop_code) + C(beds) + baths + log_sqft + floor + C(lease_year, Treatment({INDEX_BASE_YEAR}))",
                    data=d).fit()
    effects = {INDEX_BASE_YEAR: 0.0}
    for name, value in model.params.items():
        if name.startswith("C(lease_year"):
            effects[int(name.split("[T.")[1].rstrip("]"))] = value
    return (np.exp(pd.Series(effects).sort_index()) * 100), model.rsquared

hedonic = {}
for regime, sub in lease_df.groupby("province_regime"):
    for measure in ["rent_contract", "rent_effective_crm"]:
        idx, r2 = hedonic_index(sub, measure)
        hedonic[(regime, measure)] = idx
        print(f"{regime:<13} {measure:<20} R² = {r2:.3f}")
hedonic = pd.DataFrame(hedonic)

In [ ]:
COMPARE_YEARS = list(range(DISPLAY_START_YEAR, FORECAST_YEAR))
qc_leases = lease_df[lease_df.province_regime == "QC_TAL"]
naive_qc = qc_leases.groupby("lease_year").rent_contract.median().pct_change() * 100
qc_pairs = yearly_growth(pairs[pairs.province_regime == "QC_TAL"], "rent_contract")
comparison = pd.DataFrame({
    "naive median": naive_qc,
    "hedonic (similar units)": hedonic[("QC_TAL", "rent_contract")].pct_change() * 100,
    "repeat sales (same units)": index_table[("QC_TAL", "rent_contract")].pct_change() * 100,
    "pairs, median (same unit)": qc_pairs["median"],
    "pairs, rent-weighted mean": qc_pairs["weighted_mean"],
}).loc[COMPARE_YEARS]

fig, ax = plt.subplots(figsize=FIG_SIZE)
comparison.plot(marker="o", ax=ax, colormap="cividis")
ax.axhline(0, color=CHART_GREY, lw=0.8); pct_axis(ax); ax.set_xlabel("")
ax.set_title("Québec buildings, contract rent: composition vs constant-unit growth"); finish(fig)

**Constats.** (1) Les trois méthodes à composition contrôlée concordent à moins de 0,7 pt près ; la médiane naïve s'en écarte jusqu'à 4 pts. (2) Le reste de l'analyse utilise les paires même logement, la seule méthode qui sépare aussi renouvellements et relocations.

## 4. Concessions

### 4.1 Valeur des concessions

Valeur des concessions en % du loyer au bail, par année de début : elles passent de ~2 % (2019) à ~9 % (2025).

In [ ]:
contract_paid = (lease_df.rent_contract * lease_df.term_months).groupby(lease_df.lease_year).sum()
concession_by_type = (-(lease_df[group_cols].multiply(1)).groupby(lease_df.lease_year).sum()
         .div(contract_paid, axis=0) * 100).loc[ANALYSIS_YEARS]
concession_by_type.columns = [c.replace("concession_", "") for c in concession_by_type.columns]
concession_by_type["total"] = concession_by_type.sum(axis=1)
concession_by_type["pct_leases_with_concession"] = lease_df.groupby("lease_year").has_concession.mean().loc[ANALYSIS_YEARS] * 100

shown = concession_by_type.loc[ANALYSIS_YEARS]
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.bar(shown.index, shown.rent_discount, color=CHART_NAVY, label="Rent discounts")
ax.bar(shown.index, shown.ancillary, bottom=shown.rent_discount, color=CHART_GOLD, label="Ancillary (parking, locker…)")
ax.bar(shown.index, shown.other, bottom=shown.rent_discount + shown.ancillary, color=CHART_SAND, label="Other")
pct_axis(ax); ax.legend(loc="upper left"); finish(fig, "fig_concessions")

### 4.2 Pourquoi la hausse au bail et la hausse encaissée divergent depuis 2024

Jusqu'en 2023, les deux hausses avancent ensemble. Ensuite, chaque nouveau bail porte plus de concessions que le précédent : l'écart entre hausse au bail et hausse encaissée d'un même logement est exactement la **hausse de la part des concessions**.

In [ ]:
concession_share = (-lease_df.concession_total / (lease_df.rent_contract * lease_df.term_months) * 100)
gap_identity = pd.DataFrame({
    "contract − effective increase (pts)": headline[("rent_contract", "median")] - headline[("rent_effective_crm", "median")],
    "concession share (%)": concession_share.groupby(lease_df.lease_year).mean(),
}).loc[DISPLAY_START_YEAR:LAST_OBSERVED_YEAR]
gap_identity["change in concession share (pts)"] = gap_identity["concession share (%)"].diff()

fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.plot(DISPLAY_YEARS, headline.loc[DISPLAY_YEARS, ("rent_contract", "median")], "-o", color=CHART_NAVY, lw=4, ms=10, label="Contract (sRent)")
ax.plot(DISPLAY_YEARS, headline.loc[DISPLAY_YEARS, ("rent_effective_crm", "median")], "-o", color=CHART_GOLD, lw=4, ms=10, label="Effective (sRentEffective)")
pct_axis(ax); ax.legend(loc="upper left"); finish(fig, "fig_bail_encaisse")

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(gap_identity.index, gap_identity["contract − effective increase (pts)"], "-o", color=CHART_NAVY, label="contract − effective increase")
ax.plot(gap_identity.index, gap_identity["change in concession share (pts)"], "--s", color=CHART_GOLD, label="change in concession share")
ax.set_ylabel("pts"); ax.set_title("The contract − effective gap tracks concessions"); ax.legend(); ax.grid(alpha=0.3)
finish(fig)

**Lecture.** En 2025 : +7,0 % au bail, mais +3,6 % encaissé ; la moitié de la hausse est rendue en concessions, aux deux tiers des avantages accessoires (stationnement, casier). On retient donc `sRentEffective`, et la prévision traitera séparément la hausse au bail et la hausse des concessions (section 7).

## 5. Renouvellements, relocations et règles

`is_renewal` (`sRenewal`) : un locataire en place qui renouvelle, ou un logement reloué à un nouveau locataire. Les deux cas ne suivent pas les mêmes règles.

- **Québec** : taux de référence du TAL (logement non chauffé), repère et non plafond. **3,1 % en 2026** (nouvelle formule fondée sur l'IPC).
- **Ontario** : ligne directrice (2,5 % en 2023–2025, **2,1 % en 2026**), mais les logements occupés pour la première fois après le 15 novembre 2018 sont **exemptés**. Premiers baux de The Met : 2023.

On ne mélange jamais les deux régimes.

In [ ]:
regulation = external["regulation"].copy()
reg_wide = regulation.pivot(index="year", columns="province", values="reference_rate_pct")

renewal_contract = (yearly_growth(pairs[pairs.is_renewal], "rent_contract", by="province_regime", years=ANALYSIS_YEARS)
                    ["median"].unstack("province_regime"))
regulation_check = pd.DataFrame({
    "QC renouvellements (médiane, bail)": renewal_contract.get("QC_TAL"),
    "taux TAL": reg_wide["Quebec"],
    "The Met renouvellements (médiane, bail)": renewal_contract.get("ON_GUIDELINE"),
    "ligne directrice Ontario": reg_wide["Ontario"],
}).loc[ANALYSIS_YEARS[0]:FORECAST_YEAR]
regulation_check["écart QC − TAL (pts)"] = (regulation_check["QC renouvellements (médiane, bail)"]
                                                  - regulation_check["taux TAL"])
regulation_check["The Met au-dessus de la ligne directrice ?"] = (regulation_check["The Met renouvellements (médiane, bail)"]
                                                         > regulation_check["ligne directrice Ontario"])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(regulation_check.index, regulation_check["QC renouvellements (médiane, bail)"], "-o", color=CHART_NAVY, label="Équinoxe renewals")
axes[0].plot(regulation_check.index, regulation_check["taux TAL"], ":s", color=CHART_GOLD, label="TAL rate")
axes[0].set_title("Québec")
axes[1].plot(regulation_check.index, regulation_check["The Met renouvellements (médiane, bail)"], "-o", color=CHART_NAVY, label="The Met renewals")
axes[1].plot(regulation_check.index, regulation_check["ligne directrice Ontario"], ":s", color=CHART_GOLD, label="Ontario guideline")
axes[1].set_title("Ontario: The Met is exempt")
for ax in axes: pct_axis(ax); ax.legend()
finish(fig)

Renouvellements et nouveaux locataires, par régime, au bail et encaissé :

In [ ]:
SPLIT_YEARS = range(2021, 2026)
split = {}
for measure in ["rent_contract", "rent_effective_crm"]:
    t = yearly_growth(pairs, measure, by=["province_regime", "is_renewal"], years=SPLIT_YEARS)["median"].unstack(["province_regime", "is_renewal"])
    t.columns = [f"{reg}|{'renouvellement' if ren else 'relocation'}" for reg, ren in t.columns]
    split[measure] = t
split_table = pd.concat(split, axis=1)

fig, ax = plt.subplots(figsize=FIG_SIZE)
shown = list(SPLIT_YEARS)
ax.plot(shown, split_table.loc[shown, ("rent_contract", "QC_TAL|renouvellement")], "-o", color=CHART_NAVY, lw=4, ms=10, label="Renewal")
ax.plot(shown, split_table.loc[shown, ("rent_contract", "QC_TAL|relocation")], "-o", color=CHART_GOLD, lw=4, ms=10, label="New tenant")
ax.plot(shown, reg_wide.loc[shown, "Quebec"], ":s", color=CHART_GREY, lw=3, ms=8, label="TAL rate")
ax.set_xticks(shown); pct_axis(ax); ax.legend(loc="upper left"); finish(fig, "fig_renouv_reloc")
display(split_table.round(2))

The Met : premier bail après la date d'exemption ontarienne, et part de ses renouvellements au-dessus de la ligne directrice.

In [ ]:
met = pairs[(pairs.province_regime == "ON_GUIDELINE") & pairs.is_renewal]
share_above = (met.groupby("lease_year")
               .apply(lambda d: (d.growth_rent_contract > reg_wide.loc[d.name, "Ontario"]).mean() * 100, include_groups=False))
first_met_lease = lease_df.loc[lease_df.province_regime == "ON_GUIDELINE", "lease_start"].min()
exemption_date = pd.Timestamp(ONTARIO_EXEMPTION_DATE)
print(f"The Met : premier bail le {first_met_lease.date()}, après la date d'exemption {exemption_date.date()} : "
      f"{first_met_lease > exemption_date}")
print("Part des renouvellements de The Met au-dessus de la ligne directrice (%) :", share_above.round(1).to_dict())

Les renouvellements québécois suivent le taux du TAL avec un écart variable ; ceux de The Met dépassent largement la ligne directrice, ce qui est cohérent avec l'exemption. The Met est donc prévu avec le marché d'Ottawa (exemption à confirmer avec Équinoxe). Les renouvellements menaient jusqu'en 2023, les nouveaux locataires depuis 2024 ; les concessions vont surtout aux renouvellements.

## 6. Données publiques : réconciliation

### 6.1 Repères publics et tendance interne

Chaque mesure interne est rapprochée de son équivalent public : SCHL à échantillon constant (zone de chaque immeuble), IPC loyers, TAL / Ontario, hausses de renouvellement et de relocation des FPI concurrentes.

In [ ]:
cmhc_zone = external["cmhc_zone"]
building_map = external["cmhc_building_map"]
fixed = cmhc_zone[(cmhc_zone.measure == "fixed_sample_change_pct") & (cmhc_zone.bedroom == "Total")]

def cmhc_fixed_for(building, year):
    # Fixed-sample change for the building's zone, else sub-area, else CMA (October of `year`).
    row = building_map[building_map.building == building].iloc[0]
    for level in ["zone", "sub_area", "cma"]:
        hit = fixed[(fixed.geo_file == row.geo_file) & (fixed.zone == row[level]) & (fixed.year == year)]
        if len(hit) and pd.notna(hit.value.iloc[0]):
            return hit.value.iloc[0], level
    return np.nan, None

market_rows = []
for year in range(DISPLAY_START_YEAR, FORECAST_YEAR):
    for building in building_map.building:
        value, level = cmhc_fixed_for(building, year)
        market_rows.append({"lease_year": year, "building": building, "cmhc_fixed_pct": value, "cmhc_level": level})
market_by_building = pd.DataFrame(market_rows)

internal_by_building = (pairs.groupby(["lease_year", "building"])
                        .agg(equinoxe_contract=("growth_rent_contract", "median"),
                             equinoxe_effective=("growth_rent_effective_crm", "median"),
                             pairs=("lease_id", "size")).reset_index())
reconcile = internal_by_building.merge(market_by_building, on=["lease_year", "building"], how="inner")
reconcile["spread_contract_vs_cmhc"] = reconcile.equinoxe_contract - reconcile.cmhc_fixed_pct
print("Niveau SCHL utilisé :", reconcile.cmhc_level.value_counts().to_dict())

spread_by_building = reconcile.pivot_table(index="lease_year", columns="building", values="spread_contract_vs_cmhc")

fig, ax = plt.subplots(figsize=(9, 3.8))
spread_by_building.plot(ax=ax, marker="o", colormap="cividis")
ax.axhline(0, color=CHART_GREY, lw=1); ax.set_ylabel("Équinoxe − CMHC (pts)"); ax.set_xlabel("")
ax.set_title("Équinoxe contract increase vs CMHC in its zone"); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
finish(fig)

Côte à côte : Équinoxe, TAL, FPI concurrentes, SCHL (avec / sans rotation) et IPC loyers.

In [ ]:
cpi = external["cpi_annual"]
cpi_rent = cpi[cpi.series.isin(["rent_quebec", "rent_ontario"])].pivot(index="year", columns="series", values="yoy_avg_pct")
reits = external["reits"]
reit_pivot = reits.pivot_table(index="year", columns=["reit", "metric"], values="value")
cmhc_turn = external["cmhc_turnover"]
cmhc_tv = (cmhc_turn[(cmhc_turn.bedroom == "2 Bedroom") & cmhc_turn.centre.isin(["Montréal CMA", "Ottawa-Gatineau CMA (Ont. part)"])]
           .pivot(index="year", columns="centre", values=["turnover_change_pct", "non_turnover_change_pct"]))

seg = {}
for (regime, renewal), d in pairs.groupby(["province_regime", "is_renewal"]):
    seg[f"{regime} {'renewal' if renewal else 'relocation'}"] = d.groupby("lease_year").growth_rent_contract.median()
seg = pd.DataFrame(seg)

side_by_side = pd.DataFrame({
    "EQX QC renewal": seg["QC_TAL renewal"], "TAL rate": reg_wide["Quebec"],
    "CAPREIT renewal": reit_pivot.get(("CAPREIT", "renewal_increase_pct")),
    "Killam renewal": reit_pivot.get(("Killam", "renewal_increase_pct")),
    "CMHC Mtl sitting tenants": cmhc_tv.get(("non_turnover_change_pct", "Montréal CMA")),
    "EQX QC new tenant": seg["QC_TAL relocation"],
    "CAPREIT turnover": reit_pivot.get(("CAPREIT", "turnover_increase_pct")),
    "Killam turnover": reit_pivot.get(("Killam", "turnover_increase_pct")),
    "CMHC Mtl turnover": cmhc_tv.get(("turnover_change_pct", "Montréal CMA")),
    "CPI rent QC": cpi_rent["rent_quebec"], "CPI rent ON": cpi_rent["rent_ontario"],
}).loc[DISPLAY_START_YEAR:LAST_OBSERVED_YEAR]

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
groups = {"Renewals": ["EQX QC renewal", "TAL rate", "CAPREIT renewal", "Killam renewal", "CMHC Mtl sitting tenants"],
          "New tenants": ["EQX QC new tenant", "CAPREIT turnover", "Killam turnover", "CMHC Mtl turnover"]}
for ax, (title, columns) in zip(axes, groups.items()):
    for column in columns:
        if column in side_by_side and side_by_side[column].notna().any():
            is_equinoxe = column.startswith("EQX")
            ax.plot(side_by_side.index, side_by_side[column], "-o" if is_equinoxe else "--.", lw=3 if is_equinoxe else 1.5,
                    color=CHART_GOLD if is_equinoxe else None, label=column)
    ax.set_title(title); ax.legend(fontsize=8)
for ax in axes: pct_axis(ax)
finish(fig)

**Lecture.** Les renouvellements restent proches du repère réglementaire et des FPI (1–4 %). Les relocations d'Équinoxe (2–5 %) sont bien sous les gains de relocation du marché en 2022–2024 (15–28 %), puis rattrapent en 2025 alors que le marché ralentit. Les séries publiques servent donc de **repères avec un écart**, pas de substituts.

### 6.2 Kaggle : un modèle de prix appris sur les autres propriétaires, appliqué à Équinoxe

Le jeu Kaggle (≈ 18 000 annonces nettoyées, juin 2024) est une photo : il ne donne pas de **hausse**, mais il donne le **prix du marché**. On l'utilise en deux temps :

1. **Comparables** : appartements et condos non meublés à long terme, à moins de `COMPARABLE_RADIUS_KM` d'un immeuble, en retirant les annonces d'Équinoxe elles-mêmes (même endroit et même couple chambres × superficie qu'un de ses logements : c'est l'« indice » du sujet).
2. **Modèle hédonique** entraîné sur ces comparables (chambres, superficie, secteur), appliqué aux loyers affichés d'Équinoxe en juin 2024 : écart de chaque immeuble au prix du marché.

On rapproche ensuite cet écart de la hausse obtenue par les nouveaux locataires en 2025 : un immeuble sous le marché a de la marge pour monter à la relocation.

In [ ]:
KAGGLE_SNAPSHOT_MONTH = pd.Timestamp("2024-06-01")
COMPARABLE_RADIUS_KM = 3.0
OWN_LISTING_RADIUS_KM = 0.3          # annonce au même endroit qu'un immeuble d'Équinoxe
MIN_COMPARABLES = 30                 # en dessous, l'écart au marché est peu fiable
EARTH_RADIUS_KM = 6371.0
MAX_BEDS = 3

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

kaggle_path = EXTERNAL_DIR / "kaggle_listings_clean.csv"
if not kaggle_path.exists():
    print("kaggle_listings_clean.csv absent : télécharger le zip Kaggle puis public_data.build_kaggle()")
else:
    kaggle = pd.read_csv(kaggle_path)
    apartments = kaggle[kaggle.property_type.isin(["Apartment", "Condo Unit"]) & kaggle.is_long_term & ~kaggle.is_furnished
                        & kaggle.sqft.notna() & kaggle.beds.between(0, MAX_BEDS)].copy()
    apartments[["area", "distance_km"]] = [None, np.inf]
    apartments["is_equinoxe"] = False
    building_centres = unit_df.groupby("building")[["latitude", "longitude"]].median()
    for building, centre in building_centres.iterrows():
        distance = haversine_km(apartments.latitude, apartments.longitude, centre.latitude, centre.longitude)
        unit_shapes = set(zip(unit_df.loc[unit_df.building == building, "beds"].astype(float),
                              unit_df.loc[unit_df.building == building, "sqft"].round()))
        same_shape = pd.Series(list(zip(apartments.beds, apartments.sqft.round())), index=apartments.index).isin(unit_shapes)
        apartments.loc[(distance <= OWN_LISTING_RADIUS_KM) & same_shape, "is_equinoxe"] = True
        closest = (distance <= COMPARABLE_RADIUS_KM) & (distance < apartments.distance_km)
        apartments.loc[closest, "area"] = building
        apartments.loc[closest, "distance_km"] = distance[closest]
    comparables = apartments[apartments.area.notna() & ~apartments.is_equinoxe].assign(
        log_rent=lambda d: np.log(d.asking_rent), log_sqft=lambda d: np.log(d.sqft))
    market_model = smf.ols("log_rent ~ C(beds) + log_sqft + C(area)", comparables).fit()
    print(f"{len(comparables):,} comparables (annonces d'Équinoxe retirées : {apartments.is_equinoxe.sum()}) ; "
          f"R² du modèle de marché = {market_model.rsquared:.2f}".replace(".", ","))

    equinoxe_asking = asking_df[asking_df.month == KAGGLE_SNAPSHOT_MONTH].assign(
        area=lambda d: d.building, beds=lambda d: d.sBeds.clip(upper=MAX_BEDS).astype(float), log_sqft=lambda d: np.log(d.sSqft))
    equinoxe_asking["market_rent"] = np.exp(market_model.predict(equinoxe_asking))
    equinoxe_asking["gap_to_market_pct"] = (equinoxe_asking.asking_rent / equinoxe_asking.market_rent - 1) * 100
    relocation_2025 = (pairs[(pairs.lease_year == LAST_OBSERVED_YEAR) & ~pairs.is_renewal]
                       .groupby("building").growth_rent_contract.median())
    market_gap = pd.DataFrame({
        "comparables": comparables.groupby("area").size(),
        "écart au marché, juin 2024 (%)": equinoxe_asking.groupby("building").gap_to_market_pct.median(),
        f"hausse nouveaux locataires {LAST_OBSERVED_YEAR} (%)": relocation_2025,
    }).rename_axis("immeuble")
    market_gap["fiable"] = market_gap.comparables >= MIN_COMPARABLES
    reliable = market_gap[market_gap.fiable]
    print("Corrélation (immeubles fiables) entre écart au marché et hausse des nouveaux locataires : "
          f"{reliable.iloc[:, 1].corr(reliable.iloc[:, 2], method='spearman'):+.2f}".replace(".", ","))
    display(market_gap.sort_values("écart au marché, juin 2024 (%)").round(1))

    gap = market_gap.sort_values("écart au marché, juin 2024 (%)")
    names = [f"{name}{'' if ok else ' *'}" for name, ok in zip(gap.index.str.replace("Levesque", "Lévesque").str.replace("Saint-Elzear", "Saint-Elzéar"), gap.fiable)]
    values = gap["écart au marché, juin 2024 (%)"]
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    ax.barh(names, values, color=[(CHART_GOLD if v < 0 else CHART_NAVY) if ok else CHART_SAND for v, ok in zip(values, gap.fiable)])
    for yi, v in enumerate(values):
        ax.text(v + (1 if v >= 0 else -1), yi, f"{v:+.0f}%", va="center", ha="left" if v >= 0 else "right", color=CHART_NAVY)
    ax.axvline(0, color=CHART_NAVY, lw=1.5); ax.set_xlim(values.min() - 10, values.max() + 12); pct_axis(ax, "x")
    ax.set_xlabel("Équinoxe asking rent vs predicted market rent (June 2024); * fewer than 30 comparables")
    finish(fig, "fig_marche")

**Lecture.** Le modèle appris sur les autres propriétaires explique 71 % des écarts de loyer entre annonces. Appliqué à Équinoxe : les immeubles québécois affichent **7 à 19 % sous le marché** (Le Carlyle, Westpark, Saint-Elzéar), The Met **27 % au-dessus**. Pourtant, en 2025, les nouveaux locataires ont subi à peu près la même hausse partout (8 à 10 %), quel que soit l'écart au marché (corrélation +0,2 sur 4 immeubles). Deux conclusions :

- c'est une confirmation indépendante que la hausse est **une décision commune au portefeuille**, pas un rattrapage immeuble par immeuble : l'écart au marché n'est donc pas une variable de la prévision ;
- au Québec, la marge sous le marché laisse de la place aux hausses de relocation prévues (+7,6 % au bail) ; The Met, déjà au-dessus, est prévu avec un repère plus bas (marché d'Ottawa ralenti). Lévesque et Daniel-Johnson ont trop peu de comparables (< 30) pour conclure.

## 7. Prévision et backtest

### 7.1 Méthode d'évaluation (fixée avant de comparer)
- Prévision au **31 décembre de l'année précédente**, fenêtre croissante, années testées **2023, 2024, 2025**.
- Seules les données **publiées** à cette date sont utilisées (IPC filtré sur sa date de publication), **sauf le taux du TAL de l'année visée** (« TAL connu ») : publié en janvier, il est connu avant les avis de renouvellement (3 à 6 mois avant les baux du 1er juillet). C'est l'information dont dispose le propriétaire quand il fixe ses hausses ; la variante « TAL de l'an dernier » est testée (section 7.9 : 4,4 % en 2026).
- Cible : hausse encaissée du portefeuille (4 segments pondérés par leurs baux). Mesure principale : erreur absolue moyenne (MAE), en points.
- **Référence à battre** : « même hausse que l'an dernier ».
- Écart < 0,3 pt = égalité (3 années de test seulement). En cas d'égalité, l'information connue sur l'année prévue départage.
- On teste aussi la **procédure de sélection** elle-même (choix fait avec les seules années antérieures).

**Constat qui guide les modèles :** une décision annuelle commune à tout le portefeuille

La hausse au bail est presque identique dans tous les immeubles et tous les trimestres d'une même année. Et l'écart entre hausse au bail et hausse encaissée égale la hausse de la part des concessions. La prévision se décompose donc en deux questions : **quelle hausse au bail** et **combien les concessions vont encore grossir**.

In [ ]:
by_quarter = (pairs[pairs.lease_year >= BACKTEST_YEARS[0]].assign(quarter=pairs.lease_start.dt.to_period("Q"))
              .groupby("quarter").growth_rent_contract.median().round(2))
by_building = (pairs[pairs.lease_year >= BACKTEST_YEARS[0]].groupby(["lease_year", "building"]).growth_rent_contract.median()
               .unstack().round(2))
display(by_building)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(by_quarter.index.astype(str), by_quarter.values, "-o", color=CHART_NAVY)
ax.set_title("Contract increase by quarter: one step per year"); pct_axis(ax); ax.tick_params(axis="x", rotation=45)
finish(fig)

### 7.2 Entraînement de tous les modèles candidats

Tous les modèles sont entraînés ici, sans afficher de résultat ; les comparaisons sont regroupées en 7.3.

| Famille | Modèles | Paramètres estimés |
|---|---|---|
| Référence | même hausse que l'an dernier | 0 |
| Séries temporelles | moyenne 2 ans, moyenne longue, lissage exponentiel, AR(1), ARIMA | 0 à 3 |
| Repères | TAL / SCHL, IPC, FPI, plus l'écart habituel d'Équinoxe | 1 |
| Habitude + repère | w × hausse de l'an dernier + (1 − w) × (repère + écart), w ∈ {¼, ½, ¾} | 1 |
| Apprentissage par logement | Ridge, XGBoost | 20 à ~300 |
| Autres marchés | modèle entraîné sur 55 villes (SCHL) | 5 |
| Prix du logement | modèle hédonique, XGBoost de prix, hybride « prix du marché » | ~20 à ~300 |

Chaque modèle de hausse au bail est combiné à 7 règles pour la hausse des concessions. Les loyers affichés ont aussi été testés comme signal avancé (écart affiché / en place stable à ~2 % chaque année) : non retenus.

Variables connues à chaque date de prévision :

In [ ]:
cpi_monthly = external["cpi_monthly"]
SEGMENTS = [("QC_TAL", False), ("QC_TAL", True), ("ON_GUIDELINE", False), ("ON_GUIDELINE", True)]
CPI_SERIES_BY_REGIME = {"QC_TAL": "rent_quebec", "ON_GUIDELINE": "rent_ontario"}
CMHC_CMA_BY_REGIME = {"QC_TAL": ("montreal", "Montréal CMA"), "ON_GUIDELINE": ("ottawa", "Ottawa CMA")}
REG_PROVINCE_BY_REGIME = {"QC_TAL": "Quebec", "ON_GUIDELINE": "Ontario"}
MONTHS_PER_YEAR = 12

# StatCan's `releaseTime` is the ORIGINAL release only for recent points; points before mid-2021 carry the
# date of a 2021 re-release. Where the stored date is more than REPUBLISH_THRESHOLD_DAYS after the reference
# month, we use the usual publication lag instead (observed 43-49 days after the month starts).
PUBLICATION_LAG_DAYS = 50
REPUBLISH_THRESHOLD_DAYS = 60
lag_days = (cpi_monthly.released_at - cpi_monthly.month).dt.days
cpi_monthly["published_on"] = np.where(lag_days > REPUBLISH_THRESHOLD_DAYS,
                                       cpi_monthly.month + pd.Timedelta(days=PUBLICATION_LAG_DAYS),
                                       cpi_monthly.released_at)

def cpi_trailing_yoy(series, origin):
    # Year-over-year % of the latest month published on or before `origin`.
    s = cpi_monthly[(cpi_monthly.series == series) & (cpi_monthly.published_on <= origin)].sort_values("month")
    latest = s.iloc[-1]
    year_before = s[s.month == latest.month - pd.DateOffset(years=1)]
    return (latest.index_value / year_before.index_value.iloc[0] - 1) * 100

def cmhc_fixed_cma(regime, year):
    geo, name = CMHC_CMA_BY_REGIME[regime]
    hit = fixed[(fixed.geo_file == geo) & (fixed.zone == name) & (fixed.year == year)]
    return hit.value.iloc[0] if len(hit) else np.nan

def regulatory_rate(regime, year):
    v = reg_wide.loc[year, REG_PROVINCE_BY_REGIME[regime]] if year in reg_wide.index else np.nan
    return v

def segment_actuals(pairs_df, measure):
    g = pairs_df.groupby(["province_regime", "is_renewal", "lease_year"])
    return pd.DataFrame({"median": g[f"growth_{measure}"].median(), "pairs": g.size()})

actual_contract = segment_actuals(pairs, "rent_contract")
actual_effective = segment_actuals(pairs, "rent_effective_crm")

def year_features(regime, renewal, year):
    # Everything known on 31 Dec (year-1) about segment (regime, renewal), plus the regulatory rate of `year`.
    origin = pd.Timestamp(f"{year - 1}-12-31")
    def own(table, y):
        key = (regime, renewal, y)
        return table["median"].get(key, np.nan)
    concession_gap = own(actual_contract, year - 1) - own(actual_effective, year - 1)
    return {
        "province_regime": regime, "is_renewal": renewal, "lease_year": year,
        "own_contract_lag1": own(actual_contract, year - 1),
        "own_effective_lag1": own(actual_effective, year - 1),
        "own_contract_lag2": own(actual_contract, year - 2),
        "concession_gap_lag1": concession_gap,
        "cpi_rent_trailing": cpi_trailing_yoy(CPI_SERIES_BY_REGIME[regime], origin),
        "cmhc_fixed_lag1": cmhc_fixed_cma(regime, year - 1),
        "reg_rate_known": regulatory_rate(regime, year),
        "reg_rate_lagged": regulatory_rate(regime, year - 1),
    }

FORECAST_YEARS = list(range(2021, 2027))
features = pd.DataFrame([year_features(r, n, y) for y in FORECAST_YEARS for r, n in SEGMENTS])
features["actual_contract"] = features.apply(lambda r: actual_contract["median"].get((r.province_regime, r.is_renewal, r.lease_year), np.nan), axis=1)
features["actual_effective"] = features.apply(lambda r: actual_effective["median"].get((r.province_regime, r.is_renewal, r.lease_year), np.nan), axis=1)
features["pairs"] = features.apply(lambda r: actual_contract["pairs"].get((r.province_regime, r.is_renewal, r.lease_year), 0), axis=1)

Modèles de base : référence, moyenne, repères.

In [ ]:
SPREAD_WINDOW_YEARS = 3          # écart habituel = moyenne des 3 dernières années disponibles

def past_spread(segment_frame, actual_col, driver_col, year, window=SPREAD_WINDOW_YEARS):
    past = segment_frame[(segment_frame.lease_year < year)].dropna(subset=[actual_col, driver_col]).tail(window)
    return (past[actual_col] - past[driver_col]).mean() if len(past) else np.nan

def forecast_segment(seg_rows, year, method, reg_col="reg_rate_known"):
    # seg_rows: the feature rows of one segment (all years). Returns (contract, effective) for `year`.
    row = seg_rows[seg_rows.lease_year == year].iloc[0]
    past = seg_rows[seg_rows.lease_year < year]
    if method == "M0 persistence":
        return row.own_contract_lag1, row.own_effective_lag1
    if method == "M1 2-year mean":
        last2 = past.dropna(subset=["actual_contract"]).tail(2)
        return last2.actual_contract.mean(), last2.actual_effective.mean()
    if method.startswith("M2 rules+market"):
        regime_is_controlled = row.province_regime == "QC_TAL"      # The Met: exempt -> market driver
        if row.is_renewal and regime_is_controlled:
            driver_col = reg_col
        else:
            driver_col = "cmhc_fixed_lag1"
        hist = seg_rows.assign(driver=seg_rows[driver_col])
        spread = past_spread(hist, "actual_contract", "driver", year)
        if np.isnan(spread):          # no own history yet (The Met before 2024): borrow the Québec segment's spread
            return np.nan, np.nan
        contract = row[driver_col] + spread
        effective = contract - row.concession_gap_lag1
        return contract, effective
    raise ValueError(method)

def run_segment_method(method, reg_col="reg_rate_known", years=FORECAST_YEARS):
    out = []
    for (regime, renewal), rows in features.groupby(["province_regime", "is_renewal"]):
        rows = rows.sort_values("lease_year")
        for y in years:
            if y not in set(rows.lease_year):
                continue
            c, e = forecast_segment(rows, y, method, reg_col)
            out.append({"province_regime": regime, "is_renewal": renewal, "lease_year": y,
                        "pred_contract": c, "pred_effective": e})
    return pd.DataFrame(out)

segment_predictions = {}
for m in ["M0 persistence", "M1 2-year mean", "M2 rules+market"]:
    segment_predictions[m] = run_segment_method(m)
segment_predictions["M2 rules+market (reg lagged)"] = run_segment_method("M2 rules+market", reg_col="reg_rate_lagged")

Modèles par logement : Ridge et XGBoost.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import xgboost as xgb

RIDGE_ALPHA = 10.0
XGB_PARAMS = dict(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=10, random_state=0)
FIRST_TRAIN_YEAR = 2020           # first year with all lagged drivers available
YEAR_FEATURES = ["own_contract_lag1", "own_effective_lag1", "concession_gap_lag1",
                 "cpi_rent_trailing", "cmhc_fixed_lag1", "reg_rate_known"]

lease_by_id = lease_df.set_index("lease_id")
prev_concession_share = (lease_by_id.concession_total / (lease_by_id.rent_contract * lease_by_id.term_months))

def unit_design(pair_rows):
    d = pair_rows.copy()
    d["prev_lease_id"] = d.lease_id - 1          # lease_df is sorted by unit then start: previous lease = id - 1
    d["prev_concession_share"] = d.prev_lease_id.map(prev_concession_share).fillna(0)
    d["log_prev_rent"] = np.log(d.prev_rent_contract)
    d["is_ontario"] = (d.province_regime == "ON_GUIDELINE").astype(float)
    d["renewal"] = d.is_renewal.astype(float)
    d = d.merge(features[["province_regime", "is_renewal", "lease_year"] + YEAR_FEATURES],
                on=["province_regime", "is_renewal", "lease_year"], how="left")
    building_dummies = pd.get_dummies(d.prop_code, prefix="b", dtype=float)
    X = pd.concat([d[["renewal", "is_ontario", "beds", "log_prev_rent", "prev_concession_share", "lease_seq"] + YEAR_FEATURES],
                   building_dummies], axis=1)
    return d, X

assert (lease_df.groupby(["prop_code", "unit_code"]).lease_id.diff().dropna() == 1).all()
model_pairs, X_all = unit_design(pairs[pairs.lease_year >= FIRST_TRAIN_YEAR])

def run_unit_model(kind, target_measure, years=range(FIRST_TEST_YEAR, FORECAST_YEAR)):
    y_all = model_pairs[f"growth_{target_measure}"]
    out = []
    for year in years:
        train = (model_pairs.lease_year < year) & X_all.notna().all(axis=1)
        test = (model_pairs.lease_year == year)
        X_train = X_all[train].fillna(0); X_test = X_all[test].reindex(columns=X_train.columns).fillna(0)
        if kind == "ridge":
            model = make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA))
        else:
            model = xgb.XGBRegressor(**XGB_PARAMS)
        model.fit(X_train, y_all[train])
        pred = pd.Series(model.predict(X_test), index=X_test.index)
        summary = (model_pairs[test].assign(pred=pred)
                   .groupby(["province_regime", "is_renewal"]).pred.median().rename(f"pred_{target_measure}"))
        out.append(summary.reset_index().assign(lease_year=year))
    return pd.concat(out, ignore_index=True)

for kind, label in [("ridge", "M3 ridge (unit)"), ("xgb", "M4 xgboost (unit)")]:
    c = run_unit_model(kind, "rent_contract").rename(columns={"pred_rent_contract": "pred_contract"})
    e = run_unit_model(kind, "rent_effective_crm").rename(columns={"pred_rent_effective_crm": "pred_effective"})
    segment_predictions[label] = c.merge(e, on=["province_regime", "is_renewal", "lease_year"])

Modèle entraîné sur 55 villes canadiennes (SCHL), appliqué à Équinoxe.

In [ ]:
centre_panel = external["cmhc_centre"].sort_values(["centre", "year"]).copy()
for col in ["fixed_sample_change_pct_2bed", "vacancy_pct", "turnover_rate_pct", "avg_rent_2bed"]:
    centre_panel[f"{col}_lag1"] = centre_panel.groupby("centre")[col].shift(1)
centre_panel["log_rent_lag1"] = np.log(centre_panel.avg_rent_2bed_lag1)
CROSS_FEATURES = ["fixed_sample_change_pct_2bed_lag1", "vacancy_pct_lag1", "turnover_rate_pct_lag1", "log_rent_lag1"]
CENTRE_BY_REGIME = {"QC_TAL": "Montréal CMA", "ON_GUIDELINE": "Ottawa-Gatineau CMA (Ont. part)"}

def cross_market_forecast(year):
    data = centre_panel.dropna(subset=CROSS_FEATURES)
    train = data[(data.year < year)].dropna(subset=["fixed_sample_change_pct_2bed"])
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(train[CROSS_FEATURES], train.fixed_sample_change_pct_2bed)
    preds = {}
    for regime, centre in CENTRE_BY_REGIME.items():
        row = data[(data.centre == centre) & (data.year == year)]
        if len(row) == 0:                           # 2026: build next-year features from the 2025 survey
            last = centre_panel[(centre_panel.centre == centre) & (centre_panel.year == year - 1)].iloc[0]
            row = pd.DataFrame([{CROSS_FEATURES[0]: last.fixed_sample_change_pct_2bed, CROSS_FEATURES[1]: last.vacancy_pct,
                                 CROSS_FEATURES[2]: last.turnover_rate_pct, CROSS_FEATURES[3]: np.log(last.avg_rent_2bed)}])
        preds[regime] = float(model.predict(row[CROSS_FEATURES])[0])
    return preds, len(train)

market_actual = {r: centre_panel[centre_panel.centre == c].set_index("year").fixed_sample_change_pct_2bed for r, c in CENTRE_BY_REGIME.items()}
rows = []
for year in range(FIRST_TEST_YEAR, FORECAST_YEAR + 1):
    preds, n_train = cross_market_forecast(year)
    for (regime, renewal), seg_rows in features.groupby(["province_regime", "is_renewal"]):
        seg_rows = seg_rows.set_index("lease_year")
        past_years = [y for y in seg_rows.index if y < year and pd.notna(seg_rows.loc[y, "actual_contract"]) and y in market_actual[regime].index]
        past_years = past_years[-SPREAD_WINDOW_YEARS:]
        if not past_years:
            continue
        spread = np.mean([seg_rows.loc[y, "actual_contract"] - market_actual[regime][y] for y in past_years])
        contract = preds[regime] + spread
        effective = contract - seg_rows.loc[year, "concession_gap_lag1"]
        rows.append({"province_regime": regime, "is_renewal": renewal, "lease_year": year,
                     "pred_contract": contract, "pred_effective": effective, "market_pred": preds[regime], "train_rows": n_train})
segment_predictions["M5 cross-market"] = pd.DataFrame(rows)[["province_regime", "is_renewal", "lease_year", "pred_contract", "pred_effective"]]

Fonctions communes : paires depuis la table brute, poids des segments 2026, modèle en deux parties.

In [ ]:
SPREAD_SEARCH_YEARS = 10      # années remontées pour trouver les écarts au repère
def prepare_pairs(leases_raw):
    # Table des baux brute -> paires même logement (règles de la section 3).
    df = leases_raw.drop(columns=[c for c in ["sAvailable", "year"] if c in leases_raw]).rename(columns=LEASE_COLUMNS)
    for col in ["lease_start", "lease_end", "sign_date"]:
        df[col] = pd.to_datetime(df[col])
    df["is_renewal"] = df.is_renewal.astype(bool)
    df["lease_year"] = df.lease_start.dt.year
    df["province_regime"] = df.province.map(REGIME_BY_PROVINCE)
    built = build_pairs(df, measures=["rent_contract", "rent_effective_crm"])
    return df, built[built.gap_years.between(MIN_GAP_YEARS, MAX_GAP_YEARS)]

def expected_segment_leases(leases_clean, pairs_df, year):
    # Leases expected to start in `year`: current leases ending by then, split by last year's renewal share.
    current = leases_clean.sort_values("lease_start").groupby(["prop_code", "unit_code"]).tail(1)
    due = current[current.lease_end.dt.year <= year]
    renewal_share = pairs_df[pairs_df.lease_year == year - 1].groupby("province_regime").is_renewal.mean()
    weights = {}
    for regime, n in due.province_regime.value_counts().items():
        weights[(regime, True)] = n * renewal_share[regime]
        weights[(regime, False)] = n * (1 - renewal_share[regime])
    w = pd.Series(weights); w.index.names = ["province_regime", "is_renewal"]
    return w


def portfolio_figure(segment_values, weights):
    # Weighted average of segment values; an Ontario segment without a value borrows the Québec segment of the same type.
    s = segment_values.reindex(weights.index)
    for (regime, renewal) in weights.index:
        if pd.isna(s.get((regime, renewal))) and regime == "ON_GUIDELINE":
            s.loc[(regime, renewal)] = s.get(("QC_TAL", renewal))
    return float((s * weights).sum() / weights.sum())


PARTIAL_ADJUSTMENT_WEIGHT = 0.5      # share of last year's level kept; set a priori, not fitted
CONCESSION_RULES = {
    "mean2":   lambda gaps: np.nanmean(gaps[:2]) if np.isfinite(gaps[:2]).any() else 0.0,  # central
    "persist": lambda gaps: gaps[0],                   # concessions keep rising at last year's pace
    "half":    lambda gaps: gaps[0] / 2,               # the rise halves
    "zero":    lambda gaps: 0.0,                       # concession share stops rising
}

def segment_medians(pairs_df, measure):
    g = pairs_df.groupby(["province_regime", "is_renewal", "lease_year"])
    return pd.DataFrame({"median": g[f"growth_{measure}"].median(), "pairs": g.size()})

def anchor_driver(regime, renewal, year, regulation_timing="known"):
    if regime == "QC_TAL" and renewal:
        return regulatory_rate(regime, year if regulation_timing == "known" else year - 1)
    return cmhc_fixed_cma(regime, year - 1)

def two_part_segment_forecast(pairs_df, year, concession_rule="mean2", regulation_timing="known",
                              weight=PARTIAL_ADJUSTMENT_WEIGHT):
    # Forecast (contract, effective, details) for each segment in `year`, using only pairs before `year`.
    history = pairs_df[pairs_df.lease_year < year]
    contract_hist = segment_medians(history, "rent_contract")["median"]
    effective_hist = segment_medians(history, "rent_effective_crm")["median"]
    rows = []
    for regime, renewal in SEGMENTS:
        own = lambda table, y: table.get((regime, renewal, y), np.nan)
        past = [y for y in range(year - SPREAD_SEARCH_YEARS, year)
                if np.isfinite(own(contract_hist, y)) and np.isfinite(anchor_driver(regime, renewal, y, regulation_timing))]
        past = past[-SPREAD_WINDOW_YEARS:]
        spread = np.mean([own(contract_hist, y) - anchor_driver(regime, renewal, y, regulation_timing) for y in past]) if past else np.nan
        anchor = anchor_driver(regime, renewal, year, regulation_timing) + spread
        last_level = own(contract_hist, year - 1)
        contract = weight * last_level + (1 - weight) * anchor
        gaps = np.array([own(contract_hist, y) - own(effective_hist, y) for y in (year - 1, year - 2)])
        drag = CONCESSION_RULES[concession_rule](gaps)
        rows.append({"province_regime": regime, "is_renewal": renewal, "lease_year": year,
                     "last_level": last_level, "anchor_driver": anchor_driver(regime, renewal, year, regulation_timing),
                     "spread": spread, "anchor": anchor, "pred_contract": contract,
                     "concession_drag": drag, "pred_effective": contract - drag})
    return pd.DataFrame(rows)

Les 133 combinaisons : 19 modèles de hausse au bail × 7 modèles de concessions.

In [ ]:
# Cadre d'évaluation et modèles candidats (hausse au bail × hausse des concessions)
import warnings
TEST_YEARS_ALL = [FIRST_TEST_YEAR] + BACKTEST_YEARS
REQUIRED_YEARS = BACKTEST_YEARS
TIE_MARGIN_PTS = 0.30          # differences below this are treated as ties (3 test years)
SEG_KEYS = ["province_regime", "is_renewal", "lease_year"]

contract_med = actual_contract["median"]
effective_med = actual_effective["median"]
gap_med = contract_med - effective_med
seg_counts = actual_contract["pairs"]

def past_values(series, regime, renewal, year):
    s = series.xs((regime, renewal), level=["province_regime", "is_renewal"]) if (regime, renewal) in series.droplevel("lease_year").index else pd.Series(dtype=float)
    return s[s.index < year].dropna().sort_index()

# ---------- contract component models: f(regime, renewal, year) -> contract growth ----------
def c_persistence(r, n, y):
    p = past_values(contract_med, r, n, y); return p.iloc[-1] if len(p) else np.nan
def c_mean2(r, n, y):
    p = past_values(contract_med, r, n, y); return p.iloc[-2:].mean() if len(p) else np.nan
def c_expanding_mean(r, n, y):
    p = past_values(contract_med, r, n, y); return p.mean() if len(p) else np.nan

SES_ALPHA = 0.5
def c_ses(r, n, y, alpha=SES_ALPHA):
    p = past_values(contract_med, r, n, y)
    if not len(p): return np.nan
    level = p.iloc[0]
    for v in p.iloc[1:]:
        level = alpha * v + (1 - alpha) * level
    return level

AR_MIN_OBS, AR_DEFAULT_PHI = 3, 0.5
ARIMA_MIN_OBS = 4
def ar1_phi(year):
    # pooled AR(1) coefficient on deviations from each segment's own past mean, estimated on years < year
    xs, ys = [], []
    for r, n in SEGMENTS:
        p = past_values(contract_med, r, n, year)
        if len(p) < 2: continue
        mu = p.mean(); d = (p - mu).values
        xs += list(d[:-1]); ys += list(d[1:])
    if len(xs) < AR_MIN_OBS: return AR_DEFAULT_PHI
    xs, ys = np.array(xs), np.array(ys)
    return float(np.clip((xs * ys).sum() / (xs * xs).sum(), 0, 1))
def c_ar1(r, n, y):
    p = past_values(contract_med, r, n, y)
    if not len(p): return np.nan
    mu = p.mean(); return mu + ar1_phi(y) * (p.iloc[-1] - mu)

def c_arima(r, n, y):
    from statsmodels.tsa.arima.model import ARIMA
    p = past_values(contract_med, r, n, y)
    if len(p) < ARIMA_MIN_OBS: return c_persistence(r, n, y)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = ARIMA(p.values, order=(1, 0, 0)).fit()
    return float(fit.forecast(1)[0])

def cpi_anchor(r, n, y):
    return cpi_trailing_yoy(CPI_SERIES_BY_REGIME[r], pd.Timestamp(f"{y - 1}-12-31"))

REIT_FOR_SEGMENT = {True: "renewal_increase_pct", False: "turnover_increase_pct"}
def reit_anchor(r, n, y):
    vals = reits[(reits.metric == REIT_FOR_SEGMENT[n]) & (reits.year == y - 1) & reits.reit.isin(["CAPREIT", "Killam"])].value
    return vals.mean() if len(vals) else np.nan

def rules_anchor_known(r, n, y):  return anchor_driver(r, n, y, "known")
def rules_anchor_lagged(r, n, y): return anchor_driver(r, n, y, "lagged")

def anchored(anchor_fn, weight):
    # contract = weight * last level + (1 - weight) * (anchor + mean spread over the last SPREAD_WINDOW_YEARS)
    def f(r, n, y):
        p = past_values(contract_med, r, n, y)
        past = [yy for yy in p.index if np.isfinite(anchor_fn(r, n, yy))][-SPREAD_WINDOW_YEARS:]
        if not past or not np.isfinite(anchor_fn(r, n, y)): return np.nan
        spread = np.mean([p[yy] - anchor_fn(r, n, yy) for yy in past])
        target = anchor_fn(r, n, y) + spread
        return weight * p.iloc[-1] + (1 - weight) * target
    return f

def from_full_model(name):
    t = segment_predictions[name].set_index(SEG_KEYS).pred_contract
    return lambda r, n, y: t.get((r, n, y), np.nan)

CONTRACT_MODELS = {
    # name: (function, number of estimated parameters, family)
    "persistence":                    (c_persistence, 0, "benchmark"),
    "2-year mean":                    (c_mean2, 0, "time series"),
    "expanding mean":                 (c_expanding_mean, 0, "time series"),
    "exponential smoothing a=0.5":    (c_ses, 0, "time series"),
    "AR(1) mean reversion (pooled)":  (c_ar1, 2, "time series"),
    "ARIMA(1,0,0) per segment":       (c_arima, 3, "time series"),
    "rules+market anchor (TAL known)":  (anchored(rules_anchor_known, 0.0), 1, "anchor"),
    "rules+market anchor (TAL lagged)": (anchored(rules_anchor_lagged, 0.0), 1, "anchor"),
    "CPI-rent anchor":                (anchored(cpi_anchor, 0.0), 1, "anchor"),
    "REIT anchor":                    (anchored(reit_anchor, 0.0), 1, "anchor"),
    "partial adj 0.25 (TAL known)":   (anchored(rules_anchor_known, 0.25), 1, "partial adjustment"),
    "partial adj 0.50 (TAL known)":   (anchored(rules_anchor_known, 0.50), 1, "partial adjustment"),
    "partial adj 0.75 (TAL known)":   (anchored(rules_anchor_known, 0.75), 1, "partial adjustment"),
    "partial adj 0.50 (TAL lagged)":  (anchored(rules_anchor_lagged, 0.50), 1, "partial adjustment"),
    "partial adj 0.50 (CPI anchor)":  (anchored(cpi_anchor, 0.50), 1, "partial adjustment"),
    "partial adj 0.50 (REIT anchor)": (anchored(reit_anchor, 0.50), 1, "partial adjustment"),
    "XGBoost unit-level":             (from_full_model("M4 xgboost (unit)"), 300, "machine learning"),
    "Ridge unit-level":               (from_full_model("M3 ridge (unit)"), 20, "machine learning"),
    "cross-market (55 cities)":       (from_full_model("M5 cross-market"), 5, "cross-market"),
}

# ---------- concession component models: g(regime, renewal, year) -> drag (contract minus effective growth) ----------
def k_persist(r, n, y):
    p = past_values(gap_med, r, n, y); return p.iloc[-1] if len(p) else 0.0
def k_mean2(r, n, y):
    p = past_values(gap_med, r, n, y); return p.iloc[-2:].mean() if len(p) else 0.0
def k_half(r, n, y):  return k_persist(r, n, y) / 2
def k_zero(r, n, y):  return 0.0
def k_expanding(r, n, y):
    p = past_values(gap_med, r, n, y); return p.mean() if len(p) else 0.0
def k_trend(r, n, y):
    p = past_values(gap_med, r, n, y)
    if len(p) < 2: return k_persist(r, n, y)
    return max(0.0, p.iloc[-1] + (p.iloc[-1] - p.iloc[-2]))

def cma_vacancy(regime, year):
    _, name = CMHC_CMA_BY_REGIME[regime]
    centre = {"QC_TAL": "Montréal CMA", "ON_GUIDELINE": "Ottawa-Gatineau CMA (Ont. part)"}[regime]
    v = centre_panel[(centre_panel.centre == centre) & (centre_panel.year == year)].vacancy_pct
    return v.iloc[0] if len(v) else np.nan
def k_vacancy(r, n, y):
    # gap regressed on the change in CMA vacancy (Oct y-1 vs Oct y-2), pooled over segments and past years
    rows = []
    for rr, nn in SEGMENTS:
        for yy, g in past_values(gap_med, rr, nn, y).items():
            dv = cma_vacancy(rr, yy - 1) - cma_vacancy(rr, yy - 2)
            if np.isfinite(dv): rows.append((dv, g))
    dv_now = cma_vacancy(r, y - 1) - cma_vacancy(r, y - 2)
    if len(rows) < AR_MIN_OBS or not np.isfinite(dv_now): return k_persist(r, n, y)
    x, g = np.array(rows).T
    slope, intercept = np.polyfit(x, g, 1)
    return max(0.0, intercept + slope * dv_now)

CONCESSION_MODELS = {
    "persist (last year's rise)": (k_persist, 0),
    "mean of last 2 years":       (k_mean2, 0),
    "half of last year":          (k_half, 0),
    "no further rise":            (k_zero, 0),
    "expanding mean":             (k_expanding, 0),
    "trend extrapolation":        (k_trend, 0),
    "vacancy-driven regression":  (k_vacancy, 2),
}

FORECAST_GRID_YEARS = TEST_YEARS_ALL + [2026]
def component_table(models, kind):
    rows = []
    for name, (fn, *_meta) in models.items():
        for y in FORECAST_GRID_YEARS:
            for r, n in SEGMENTS:
                rows.append({"model": name, "province_regime": r, "is_renewal": n, "lease_year": y, kind: fn(r, n, y)})
    return pd.DataFrame(rows)

contract_preds = component_table(CONTRACT_MODELS, "pred_contract")
concession_preds = component_table(CONCESSION_MODELS, "pred_drag")

# ---------- metrics ----------
print(f'{contract_preds.model.nunique()} modèles de hausse au bail × {concession_preds.model.nunique()} modèles de concessions')

Erreurs des 133 combinaisons sur les années de test.

In [ ]:
def portfolio_series(pred_by_seg, years):
    out = {}
    for y in years:
        w = seg_counts.xs(y, level="lease_year")
        p = pred_by_seg.xs(y, level="lease_year") if y in pred_by_seg.index.get_level_values("lease_year") else pd.Series(dtype=float)
        out[y] = portfolio_figure(p, w) if len(p) and p.notna().any() else np.nan
    return pd.Series(out)

def actual_portfolio(table, years):
    return pd.Series({y: portfolio_figure(table.xs(y, level="lease_year"), seg_counts.xs(y, level="lease_year")) for y in years})

ACTUAL = {"contract": actual_portfolio(contract_med, TEST_YEARS_ALL),
          "effective": actual_portfolio(effective_med, TEST_YEARS_ALL),
          "drag": actual_portfolio(gap_med, TEST_YEARS_ALL)}

def metrics(pred_seg, actual_seg, target):
    port_p = portfolio_series(pred_seg, TEST_YEARS_ALL)
    err = (port_p - ACTUAL[target])
    req = err.loc[REQUIRED_YEARS]
    prev_actual = {"contract": actual_portfolio(contract_med, [y - 1 for y in REQUIRED_YEARS]),
                   "effective": actual_portfolio(effective_med, [y - 1 for y in REQUIRED_YEARS]),
                   "drag": actual_portfolio(gap_med, [y - 1 for y in REQUIRED_YEARS])}[target].values
    direction_ok = np.sign(port_p.loc[REQUIRED_YEARS].values - prev_actual) == np.sign(ACTUAL[target].loc[REQUIRED_YEARS].values - prev_actual)
    cells = pd.concat([pred_seg.rename("p"), actual_seg.rename("a")], axis=1).dropna()
    cells = cells[cells.index.get_level_values("lease_year").isin(REQUIRED_YEARS)]
    return {"MAE": req.abs().mean(), "RMSE": np.sqrt((req ** 2).mean()), "bias": req.mean(), "max abs": req.abs().max(),
            "segment MAE": (cells.p - cells.a).abs().mean(), "direction hits": f"{direction_ok.sum()}/{len(REQUIRED_YEARS)}",
            **{f"err {y}": err.loc[y] for y in TEST_YEARS_ALL}}

def leaderboard_for(preds, value_col, actual_seg, target, meta):
    rows = []
    for name, g in preds.groupby("model", sort=False):
        s = g.set_index(SEG_KEYS)[value_col]
        rows.append({"model": name, **meta(name), **metrics(s, actual_seg, target)})
    return pd.DataFrame(rows).set_index("model")

contract_board = leaderboard_for(contract_preds, "pred_contract", contract_med, "contract",
                                 lambda m: {"params": CONTRACT_MODELS[m][1], "family": CONTRACT_MODELS[m][2]}).sort_values("MAE")
concession_board = leaderboard_for(concession_preds, "pred_drag", gap_med, "drag",
                                   lambda m: {"params": CONCESSION_MODELS[m][1]}).sort_values("MAE")

def select(board):
    # declared rule: lowest MAE; within TIE_MARGIN_PTS -> lower segment MAE, then fewer parameters
    best = board.MAE.min()
    tied = board[board.MAE <= best + TIE_MARGIN_PTS].copy()
    tied["seg_rank"] = tied["segment MAE"].rank()
    return tied.sort_values(["segment MAE", "params"]).index[0], tied

chosen_contract, contract_ties = select(contract_board)
chosen_concession, concession_ties = select(concession_board)

def combine(contract_name, concession_name):
    c = contract_preds[contract_preds.model == contract_name].set_index(SEG_KEYS).pred_contract
    k = concession_preds[concession_preds.model == concession_name].set_index(SEG_KEYS).pred_drag
    return c - k

Modèles de **prix** : prévoir le loyer de chaque logement (caractéristiques + effet de l'année), puis en déduire la hausse par rapport à son loyer précédent ; et un hybride où le loyer des nouveaux locataires suit le prix de marché du logement (nouveaux baux de l'année précédente × (1 + repère SCHL)).

In [ ]:
HEDONIC_PRICE_FORMULA = "np.log({y}) ~ C(prop_code) + C(beds) + baths + np.log(sqft) + floor + C(lease_year)"
MARKET_PRICE_FORMULA = "np.log(rent_contract) ~ C(prop_code) + C(beds) + baths + np.log(sqft) + floor"
XGB_PRICE_PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=0)
PRICE_MEASURES = {"contract": "rent_contract", "effective": "rent_effective_crm"}
MID_YEAR = "-07-01"                     # date type d'un bail de l'année prévue
price_leases, price_pairs = prepare_pairs(leases_raw)

def by_segment(values, frame):
    return values.groupby([frame.loc[values.index, "province_regime"], frame.loc[values.index, "is_renewal"]]).median()

def annualised_growth(target, level, measure):
    return ((level / target[f"prev_{measure}"]) ** (1 / target.gap_years) - 1) * 100

def hedonic_trend_price(year, measure, target):
    train = price_leases[price_leases.lease_year < year]
    model = smf.ols(HEDONIC_PRICE_FORMULA.format(y=measure), train).fit()
    effects = {int(n.split("[T.")[1].rstrip("]")): v for n, v in model.params.items() if n.startswith("C(lease_year)")}
    trend = effects.get(year - 1, 0.0) - effects.get(year - 2, 0.0)      # dernière variation de l'effet année, prolongée
    seen = target[target.prop_code.isin(train.prop_code.unique()) & target.beds.isin(train.beds.unique())]
    return annualised_growth(seen, np.exp(model.predict(seen.assign(lease_year=year - 1)) + trend), measure)

def xgboost_price(year, measure, target):
    columns = ["beds", "baths", "sqft", "floor", "lease_year"]
    train = price_leases[price_leases.lease_year < year]
    X = pd.concat([train[columns], pd.get_dummies(train.prop_code, dtype=float)], axis=1)
    model = xgb.XGBRegressor(**XGB_PRICE_PARAMS).fit(X, np.log(train[measure]))
    X_target = pd.concat([target[columns], pd.get_dummies(target.prop_code, dtype=float)], axis=1).reindex(columns=X.columns, fill_value=0)
    return annualised_growth(target, pd.Series(np.exp(model.predict(X_target)), index=target.index), measure)

def market_price_relocations(year, target):
    recent = price_leases[(price_leases.lease_year == year - 1) & ~price_leases.is_renewal]
    model = smf.ols(MARKET_PRICE_FORMULA, recent).fit()
    new = target[~target.is_renewal & target.prop_code.isin(recent.prop_code.unique()) & target.beds.isin(recent.beds.unique())]
    anchor = new.province_regime.map(lambda regime: anchor_driver(regime, False, year))
    return annualised_growth(new, np.exp(model.predict(new)) * (1 + anchor / 100), "rent_contract")

price_rows = []
for year in range(SCORING_START, FORECAST_YEAR):
    target = price_pairs[price_pairs.lease_year == year]
    weights = target.groupby(["province_regime", "is_renewal"]).size()
    retained_segments = two_part_segment_forecast(price_pairs, year).set_index(["province_regime", "is_renewal"])
    relocations = market_price_relocations(year, target)
    for label, measure in PRICE_MEASURES.items():
        actual = portfolio_figure(by_segment(target[f"growth_{measure}"], target), weights)
        hybrid = retained_segments[f"pred_{label}"].copy()
        for regime, values in relocations.groupby(target.loc[relocations.index, "province_regime"]):
            drag = retained_segments.loc[(regime, False), "concession_drag"] if label == "effective" else 0.0
            hybrid.loc[(regime, False)] = values.median() - drag
        predictions = {"Prix hédonique + tendance": by_segment(hedonic_trend_price(year, measure, target), target),
                       "Prix XGBoost": by_segment(xgboost_price(year, measure, target), target),
                       "Hybride : prix du marché (nouveaux loc.)": hybrid}
        for name, segments in predictions.items():
            price_rows.append({"méthode": name, "mesure": label, "année": year, "erreur": portfolio_figure(segments, weights) - actual})
price_errors = pd.DataFrame(price_rows)

latest_leases = price_leases.sort_values("lease_start").groupby(["prop_code", "unit_code"]).tail(1)
due_leases = latest_leases[latest_leases.lease_end.dt.year <= FORECAST_YEAR]
due_pairs = due_leases.assign(prev_rent_contract=due_leases.rent_contract, prev_rent_effective_crm=due_leases.rent_effective_crm,
                              lease_year=FORECAST_YEAR,
                              gap_years=((pd.Timestamp(f"{FORECAST_YEAR}{MID_YEAR}") - due_leases.lease_start).dt.days / DAYS_PER_YEAR).clip(lower=MIN_GAP_YEARS))
weights_2026 = expected_segment_leases(price_leases, price_pairs, FORECAST_YEAR)
price_forecast_2026 = {(name, label): portfolio_figure(by_segment(model_fn(FORECAST_YEAR, measure, due_pairs), due_pairs), weights_2026)
                       for name, model_fn in [("Prix hédonique + tendance", hedonic_trend_price), ("Prix XGBoost", xgboost_price)]
                       for label, measure in PRICE_MEASURES.items()}
print(f"modèles de prix évalués sur {price_errors['année'].nunique()} années")

### 7.3 Résultats et comparaisons

Tous les modèles sont jugés de la même façon (7.1). Trois questions :
1. choisir le meilleur parmi beaucoup de modèles aide-t-il ?
2. quel modèle retenir (tableau de synthèse, une ligne par famille) ?
3. le résultat tient-il sur d'autres données (35 villes) ?

**1. Test de la procédure de sélection.**

Choisir le meilleur de 133 modèles sur 3 années de test surajuste. Chaque procédure ci-dessous fait son choix **avec les seules années antérieures**, puis est évaluée sur l'année suivante.

In [ ]:
# Test de la procédure de sélection (choix fait avec les seules années antérieures)
BENCHMARK = "persistence | concessions: persist (last year's rise)"
RETAINED = "partial adj 0.50 (TAL known) | concessions: mean of last 2 years"
NESTED_TEST_YEARS = BACKTEST_YEARS
ALL_YEARS = list(range(SCORING_START, FORECAST_YEAR + 1))
TOP_K = 5

def combo_name(c, k): return f"{c} | concessions: {k}"
combo_preds = {}
for c in CONTRACT_MODELS:
    cf = CONTRACT_MODELS[c][0]
    for k in CONCESSION_MODELS:
        kf = CONCESSION_MODELS[k][0]
        rows = {(r, n, y): cf(r, n, y) - kf(r, n, y) for y in ALL_YEARS for r, n in SEGMENTS}
        combo_preds[combo_name(c, k)] = pd.Series(rows).rename_axis(SEG_KEYS)
combo_params = {combo_name(c, k): CONTRACT_MODELS[c][1] + CONCESSION_MODELS[k][1] for c in CONTRACT_MODELS for k in CONCESSION_MODELS}
combo_family = {combo_name(c, k): CONTRACT_MODELS[c][2] for c in CONTRACT_MODELS for k in CONCESSION_MODELS}

actual_eff_all = pd.Series({y: portfolio_figure(effective_med.xs(y, level="lease_year"), seg_counts.xs(y, level="lease_year"))
                            for y in range(SCORING_START, FORECAST_YEAR)})
def portfolio_pred(series, y, weights=None):
    w = seg_counts.xs(y, level="lease_year") if weights is None else weights
    p = series.xs(y, level="lease_year")
    return portfolio_figure(p, w) if p.notna().any() else np.nan
portfolio_by_combo = pd.DataFrame({name: {y: portfolio_pred(s, y) for y in range(SCORING_START, FORECAST_YEAR)} for name, s in combo_preds.items()})
errors_by_combo = portfolio_by_combo.sub(actual_eff_all, axis=0)

def pick_best(candidates, upto_year):
    past = errors_by_combo.loc[SCORING_START:upto_year - 1, candidates]
    past = past.loc[:, past.notna().all()]
    if past.shape[1] == 0: return None
    mae = past.abs().mean()
    best = mae.min()
    tied = mae[mae <= best + TIE_MARGIN_PTS].index
    return min(tied, key=lambda n: (combo_params[n], mae[n]))

def pick_topk_average(candidates, upto_year, k=TOP_K):
    past = errors_by_combo.loc[SCORING_START:upto_year - 1, candidates]
    past = past.loc[:, past.notna().all()]
    return list(past.abs().mean().nsmallest(k).index)

ALL_COMBOS = list(combo_preds)
PA_COMBOS = [n for n in ALL_COMBOS if combo_family[n] == "partial adjustment"]
TS_COMBOS = [n for n in ALL_COMBOS if combo_family[n] in ("time series", "benchmark")]
REASONABLE = [n for n in ALL_COMBOS if combo_family[n] not in ("machine learning",)]

PROCEDURES = {
    "benchmark: persistence (both parts)":       lambda y: ["persistence | concessions: persist (last year's rise)"],
    "fixed a priori: partial adj 0.5 TAL + mean2": lambda y: ["partial adj 0.50 (TAL known) | concessions: mean of last 2 years"],
    "pick best so far, all 133 combos":           lambda y: [pick_best(ALL_COMBOS, y)],
    "pick best so far, partial-adjustment family": lambda y: [pick_best(PA_COMBOS, y)],
    "pick best so far, time-series family":       lambda y: [pick_best(TS_COMBOS, y)],
    f"average of top {TOP_K} so far":             lambda y: pick_topk_average(ALL_COMBOS, y),
    "average of all non-ML combos (no selection)": lambda y: REASONABLE,
}

def procedure_forecast(members, y):
    vals = [portfolio_by_combo.loc[y, m] if y in portfolio_by_combo.index else np.nan for m in members if m is not None]
    return np.nanmean(vals) if len(vals) else np.nan

nested_rows, nested_picks = [], {}
for proc, chooser in PROCEDURES.items():
    for y in NESTED_TEST_YEARS:
        members = chooser(y)
        p = procedure_forecast(members, y)
        nested_rows.append({"procedure": proc, "year": y, "predicted": p, "actual": actual_eff_all[y], "error": p - actual_eff_all[y]})
        nested_picks[(proc, y)] = members
nested = pd.DataFrame(nested_rows)
nested_board = (nested.groupby("procedure", sort=False).error
                .agg(MAE=lambda e: e.abs().mean(), RMSE=lambda e: np.sqrt((e ** 2).mean()), bias="mean", max_abs=lambda e: e.abs().max())
                .join(nested.pivot(index="procedure", columns="year", values="error").add_prefix("err "))
                .sort_values("MAE"))

NOMS_PROCEDURES = {
    "benchmark: persistence (both parts)": "référence : même hausse que l'an dernier",
    "fixed a priori: partial adj 0.5 TAL + mean2": "fixé d'avance : ½ habitude + ½ repère (retenu)",
    "pick best so far, all 133 combos": "choisir le meilleur des 133 à chaque année",
    "pick best so far, partial-adjustment family": "choisir le meilleur des mélanges habitude + repère",
    "pick best so far, time-series family": "choisir la meilleure série temporelle",
    f"average of top {TOP_K} so far": f"moyenne des {TOP_K} meilleurs",
    "average of all non-ML combos (no selection)": "moyenne de tous les modèles (sans choix)",
}
display((nested_board.rename(index=NOMS_PROCEDURES).rename_axis("procédure")
 .rename(columns={"bias": "biais", "max_abs": "erreur max"}).round(2)))

in_sample_best = errors_by_combo.loc[REQUIRED_YEARS].abs().mean().min()
selection_bars = {"Best of 133\n(in hindsight)": in_sample_best,
                  "Same choice\n(no hindsight)": nested_board.loc["pick best so far, all 133 combos", "MAE"],
                  "Our model\n(fixed upfront)": errors_by_combo.loc[REQUIRED_YEARS, RETAINED].abs().mean(),
                  "Same increase\nas last year": errors_by_combo.loc[REQUIRED_YEARS, BENCHMARK].abs().mean()}
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.bar(list(selection_bars), list(selection_bars.values()), color=[CHART_SAND, CHART_GREY, CHART_GOLD, CHART_NAVY])
for i, v in enumerate(selection_bars.values()): ax.text(i, v + 0.03, f"{v:.2f}", ha="center", color=CHART_NAVY)
ax.set_ylabel("mean error 2023-2025 (pts)"); ax.grid(axis="y", color=CHART_LIGHT, lw=0.8)
finish(fig, "fig_surapprentissage")

**2. Tableau de synthèse** : comparaison appariée du modèle retenu avec la référence, puis un modèle représentatif par famille, modèles de prix compris.

In [ ]:
# Comparaison appariée segment × année : modèle retenu contre la référence
BOOTSTRAP_DRAWS = 5000
CONFIDENCE_PCT = 95
LOWER_PCTL, UPPER_PCTL = (100 - CONFIDENCE_PCT) / 2, 100 - (100 - CONFIDENCE_PCT) / 2
cells = pd.concat([combo_preds[BENCHMARK].rename("benchmark"), combo_preds[RETAINED].rename("two_part"),
                   effective_med.rename("actual")], axis=1).dropna()
cells = cells[cells.index.get_level_values("lease_year").isin(REQUIRED_YEARS)]
diff = (cells.two_part - cells.actual).abs() - (cells.benchmark - cells.actual).abs()
rng = np.random.default_rng(0)
boot = [rng.choice(diff.values, len(diff)).mean() for _ in range(BOOTSTRAP_DRAWS)]
print(f"{len(diff)} cellules segment × année ; erreur retenu − référence = {diff.mean():+.2f} pt, "
      f"intervalle à {CONFIDENCE_PCT} % [{np.percentile(boot, LOWER_PCTL):+.2f}, {np.percentile(boot, UPPER_PCTL):+.2f}]")

# Six modèles représentatifs (les 133 combinaisons restent calculées ci-dessus)
w_2026 = expected_segment_leases(*prepare_pairs(leases_raw), FORECAST_YEAR)
REPRESENTATIVE = {
    "Référence : même hausse que l'an dernier": ("persistence", "persist (last year's rise)"),
    "Moyenne des 2 dernières années":            ("2-year mean", "mean of last 2 years"),
    "Repères seuls (TAL / SCHL + écart)":        ("rules+market anchor (TAL known)", "mean of last 2 years"),
    "XGBoost par logement":                      ("XGBoost unit-level", "mean of last 2 years"),
    "Entraîné sur 55 villes (SCHL)":             ("cross-market (55 cities)", "mean of last 2 years"),
    "Retenu : ½ habitude + ½ repère":            ("partial adj 0.50 (TAL known)", "mean of last 2 years"),
}
def representative_row(contract_name, concession_name):
    eff = combo_preds[combo_name(contract_name, concession_name)]
    con = contract_preds[contract_preds.model == contract_name].set_index(SEG_KEYS).pred_contract
    con_err = portfolio_series(con, REQUIRED_YEARS) - ACTUAL["contract"].loc[REQUIRED_YEARS]
    def f2026(s):
        p = s.xs(FORECAST_YEAR, level="lease_year")
        return portfolio_figure(p, w_2026) if p.notna().any() else np.nan
    all_years_error = errors_by_combo[combo_name(contract_name, concession_name)].dropna()
    return {"MAE encaissé 2023-2025": errors_by_combo.loc[REQUIRED_YEARS, combo_name(contract_name, concession_name)].abs().mean(),
            f"MAE encaissé {SCORING_START}-{LAST_OBSERVED_YEAR}": all_years_error.abs().mean(),
            "années testées": len(all_years_error),
            "MAE au bail 2023-2025": con_err.abs().mean(),
            "2026 encaissé %": f2026(eff), "2026 au bail %": f2026(con)}
representative = pd.DataFrame({label: representative_row(*names) for label, names in REPRESENTATIVE.items()}).T
for name in ["Prix hédonique + tendance", "Prix XGBoost", "Hybride : prix du marché (nouveaux loc.)"]:
    errors = price_errors[price_errors["méthode"] == name]
    effective = errors[errors.mesure == "effective"].set_index("année").erreur
    contract = errors[errors.mesure == "contract"].set_index("année").erreur
    representative.loc[name] = pd.Series({
        "MAE encaissé 2023-2025": effective.loc[REQUIRED_YEARS].abs().mean(),
        f"MAE encaissé {SCORING_START}-{LAST_OBSERVED_YEAR}": effective.abs().mean(), "années testées": len(effective),
        "MAE au bail 2023-2025": contract.loc[REQUIRED_YEARS].abs().mean(),
        "2026 encaissé %": price_forecast_2026.get((name, "effective"), np.nan),
        "2026 au bail %": price_forecast_2026.get((name, "contract"), np.nan)})
display(representative.round(2))

COMPLEXITY_MODELS = {"Same increase": "persistence", "2-year mean": "2-year mean", "Our model": "partial adj 0.50 (TAL known)",
                     "55 cities": "cross-market (55 cities)", "Ridge": "Ridge unit-level", "XGBoost": "XGBoost unit-level"}
LABEL_OFFSETS = {"Same increase": (18, -38), "2-year mean": (-10, 30), "Our model": (18, -6)}
fig, ax = plt.subplots(figsize=FIG_SIZE)
for label, model in COMPLEXITY_MODELS.items():
    rule = "persist (last year's rise)" if model == "persistence" else "mean of last 2 years"
    mae = errors_by_combo.loc[REQUIRED_YEARS, combo_name(model, rule)].abs().mean()
    color = CHART_GOLD if label == "Our model" else CHART_NAVY
    ax.scatter(CONTRACT_MODELS[model][1] + 1, mae, s=220, color=color, zorder=3)
    ax.annotate(label, (CONTRACT_MODELS[model][1] + 1, mae), xytext=LABEL_OFFSETS.get(label, (14, 8)), textcoords="offset points", color=color)
ax.set_xscale("log"); ax.set_xlabel("number of estimated parameters + 1 (log scale)"); ax.set_ylabel("mean error 2023-2025 (pts)")
ax.set_ylim(bottom=0); ax.grid(color=CHART_LIGHT, lw=0.8)
finish(fig, "fig_complexite")

**3. Test sur d'autres données : 35 grandes villes canadiennes (SCHL).**

Même question, autres marchés : prévoir la hausse à échantillon constant (2 chambres) de chaque région métropolitaine pour 2022–2025, avec les seules années antérieures. Ce sont les ~35 grandes régions métropolitaines du panel SCHL (le modèle « 55 villes » utilisait aussi les provinces et petits centres). Le repère est la médiane nationale de l'année précédente plus l'écart habituel de la ville (3 ans).

In [ ]:
cma = centre_panel[centre_panel.is_cma].pivot(index="year", columns="centre", values="fixed_sample_change_pct_2bed")
EXTERNAL_TEST_YEARS = range(FIRST_TEST_YEAR, FORECAST_YEAR)
external_errors = []
for year in EXTERNAL_TEST_YEARS:
    previous, actual = cma.loc[year - 1], cma.loc[year]
    past_years = [y for y in cma.index if y < year][-SPREAD_WINDOW_YEARS:]
    usual_spread = pd.concat([cma.loc[y] - cma.loc[y].median() for y in past_years], axis=1).mean(axis=1)
    anchor = previous.median() + usual_spread
    forecasts = {"Référence : même hausse que l'an dernier": previous,
                 "Moyenne des 2 dernières années": cma.loc[[year - 1, year - 2]].mean(),
                 "Repère seul (médiane nationale + écart)": anchor,
                 "Retenu : ½ habitude + ½ repère": PARTIAL_ADJUSTMENT_WEIGHT * previous + (1 - PARTIAL_ADJUSTMENT_WEIGHT) * anchor}
    for name, f in forecasts.items():
        err = (f - actual).dropna()
        external_errors.append(pd.DataFrame({"modèle": name, "année": year, "ville": err.index, "erreur": err.values}))
external_errors = pd.concat(external_errors, ignore_index=True)
external_board = external_errors.pivot_table(index="modèle", columns="année", values="erreur", aggfunc=lambda e: e.abs().mean())
external_board.insert(0, "MAE 2022-2025", external_errors.groupby("modèle").erreur.apply(lambda e: e.abs().mean()))
external_board.insert(1, "points", external_errors.groupby("modèle").size())

paired = external_errors.pivot_table(index=["année", "ville"], columns="modèle", values="erreur").dropna().abs()
gain = paired["Retenu : ½ habitude + ½ repère"] - paired["Référence : même hausse que l'an dernier"]
rng = np.random.default_rng(0)
boot = [rng.choice(gain.values, len(gain)).mean() for _ in range(BOOTSTRAP_DRAWS)]
print(f"{len(gain)} points ville × année ; erreur retenu − référence = {gain.mean():+.2f} pt, "
      f"intervalle à {CONFIDENCE_PCT} % [{np.percentile(boot, LOWER_PCTL):+.2f}, {np.percentile(boot, UPPER_PCTL):+.2f}]")
display(external_board.sort_values("MAE 2022-2025").round(2))

shown = list(EXTERNAL_TEST_YEARS); x = np.arange(len(shown)); width = 0.38
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.bar(x - width / 2, external_board.loc["Référence : même hausse que l'an dernier", shown], width, color=CHART_NAVY, label="Same increase as last year")
ax.bar(x + width / 2, external_board.loc["Retenu : ½ habitude + ½ repère", shown], width, color=CHART_GOLD, label="Our model")
ax.set_xticks(x, shown); ax.set_ylabel("mean error (pts)"); ax.grid(axis="y", color=CHART_LIGHT, lw=0.8); ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
ax.legend(loc="upper center", ncol=2); finish(fig, "fig_villes")

**Lecture.** Sur 135 points ville × année, le modèle retenu fait une erreur moyenne de **2,38 pts** contre **2,48** pour la référence, et la bat 3 années sur 4. L'écart (−0,10 pt, intervalle [−0,22 ; +0,02]) est petit mais va dans le même sens qu'à Équinoxe : mélanger l'habitude et le repère est au moins aussi bon que répéter l'an dernier, et ça ne dépend pas d'Équinoxe.

### 7.4 Décision : un modèle simple, parce qu'il y a peu de données

- La hausse est **une décision par an** : 6 années × 4 segments, soit ~20 observations utiles. Un modèle à beaucoup de paramètres (XGBoost, Ridge) apprend le bruit de ces quelques années.
- Preuve par le surapprentissage : le meilleur des 133 modèles **choisi après coup** semble excellent, mais le même choix **fait sans connaître l'avenir** perd tout son avantage (tableau ci-dessus). Plus de modèles essayés = plus de chance, pas plus de précision.
- **Aucune procédure ne bat la référence** (1,49). Le modèle retenu (1,61) est à égalité statistique ; il n'a **qu'un paramètre**, fixé d'avance (½), donc rien à surajuster.
- **Fenêtre plus longue (2021–2025, 5 années)** : référence 1,16, modèle retenu 1,42. L'écart (0,26) reste sous le seuil d'égalité de 0,3, mais la référence reste devant. Avant 2021, trop peu de paires et pas assez d'historique pour calculer l'écart habituel au repère (3 ans) ; les années 2023–2025 sont celles demandées par le défi et les seules où les six immeubles et The Met (Ontario) sont présents.
- **Prévoir le prix plutôt que la hausse** (dernières lignes du tableau de synthèse) : le modèle hédonique a le meilleur score encaissé sur 2023–2025 (1,30) mais le pire au bail (2,95), reste derrière la référence sur 2021–2025 (1,25 contre 1,16), et prévoit 7,7 % au bail en 2026 : il ne tient que par sa tendance prolongée. XGBoost, qui ne sait pas prolonger l'année, prévoit ≈ 0 %. Les caractéristiques d'un logement s'annulent d'un bail à l'autre : la hausse vient de l'effet de l'année, d'où des modèles de **hausse**.
- Ce qui est connu de 2026 tranche : taux du TAL de 5,9 % à 3,1 % (publié en janvier 2026, avant les avis de renouvellement), ligne directrice ontarienne 2,1 %, marché d'Ottawa 3,5 %. Seul le modèle retenu l'intègre.

**Modèle retenu : ½ × hausse de l'an dernier + ½ × (repère de l'année + écart habituel), moins la hausse moyenne des concessions des deux dernières années.** La référence est la règle de mesure. Qu'elle soit difficile à battre montre qu'Équinoxe ajuste ses prix progressivement, ce que le modèle retenu formalise.

### 7.5 `estimate_2026()` et `backtest()`

Les deux fonctions prennent la table des baux brute du notebook de départ, reconstruisent les paires même logement et appliquent le modèle retenu avec les seules données antérieures à l'année visée.

In [ ]:
def estimate_2026(leases, asking, external=None, concession_rule="mean2"):
    # Hausse 2026 (%) du loyer ENCAISSÉ d'un même logement, avec la hausse au bail et le détail par segment.
    # `asking` est accepté pour respecter le gabarit mais non utilisé : signal testé et rejeté (section 7.2).
    year = FORECAST_YEAR
    leases_clean, pairs_df = prepare_pairs(leases)
    segments = two_part_segment_forecast(pairs_df, year, concession_rule).set_index(["province_regime", "is_renewal"])
    weights = expected_segment_leases(leases_clean, pairs_df, year)
    return {
        "definition": f"moyenne, pondérée par les baux attendus en {year}, des hausses médianes du loyer encaissé (sRentEffective) "
                      f"d'un même logement par rapport à son bail précédent, pour 4 segments, baux débutant en {year}",
        "effective_pct": portfolio_figure(segments.pred_effective, weights),
        "contract_pct": portfolio_figure(segments.pred_contract, weights),
        "segments": segments.assign(expected_leases=weights),
    }

def backtest(leases, target_year, concession_rule="mean2"):
    # Applique la méthode au 31 déc. (target_year − 1) et compare avec ce qui s'est passé en target_year.
    _, pairs_df = prepare_pairs(leases)
    segments = two_part_segment_forecast(pairs_df, target_year, concession_rule).set_index(["province_regime", "is_renewal"])
    realised = pairs_df[pairs_df.lease_year == target_year]
    actual_c = realised.groupby(["province_regime", "is_renewal"]).growth_rent_contract.median()
    actual_e = realised.groupby(["province_regime", "is_renewal"]).growth_rent_effective_crm.median()
    weights = realised.groupby(["province_regime", "is_renewal"]).size()
    result = {}
    for measure, actual in [("effective", actual_e), ("contract", actual_c)]:
        predicted = portfolio_figure(segments[f"pred_{measure}"], weights)
        realised_value = portfolio_figure(actual, weights)
        result[measure] = {"predicted": round(predicted, 2), "actual": round(realised_value, 2),
                           "error": round(predicted - realised_value, 2)}
    return result

backtest_results = {yr: backtest(leases_raw, yr) for yr in BACKTEST_YEARS}
display(pd.DataFrame({yr: {"encaissé prévu": r["effective"]["predicted"], "encaissé réalisé": r["effective"]["actual"],
                         "erreur encaissé": r["effective"]["error"], "au bail prévu": r["contract"]["predicted"],
                         "au bail réalisé": r["contract"]["actual"], "erreur au bail": r["contract"]["error"]}
                    for yr, r in backtest_results.items()}).T)
mae_effective = np.mean([abs(r["effective"]["error"]) for r in backtest_results.values()])
mae_contract = np.mean([abs(r["contract"]["error"]) for r in backtest_results.values()])
print(f"MAE 2023-2025 : encaissé {mae_effective:.2f} pt, au bail {mae_contract:.2f} pt".replace(".", ","))

x = np.arange(len(BACKTEST_YEARS)); width = 0.27
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.bar(x - width, [backtest_results[y]["effective"]["actual"] for y in BACKTEST_YEARS], width, color=CHART_SAND, label="Actual")
ax.bar(x, [portfolio_by_combo.loc[y, BENCHMARK] for y in BACKTEST_YEARS], width, color=CHART_NAVY, label="Same increase as last year")
ax.bar(x + width, [backtest_results[y]["effective"]["predicted"] for y in BACKTEST_YEARS], width, color=CHART_GOLD, label="Our model")
ax.set_xticks(x, BACKTEST_YEARS); pct_axis(ax); ax.set_ylabel("portfolio effective increase"); ax.set_ylim(0, ax.get_ylim()[1] * 1.3)
ax.legend(loc="upper center", ncol=3); finish(fig, "fig_backtest")

### 7.6 Estimation 2026

L'extrait garde une part fixe des logements : sa **répartition** est représentative, pas ses **volumes**. Occupation, inoccupation et absorption ne sont donc **jamais calculées** ; les nombres de baux servent uniquement de poids entre segments.

In [ ]:
estimate = estimate_2026(leases_raw, asking_raw, external)
def estimate_2026_lagged_regulation(leases):
    leases_clean, pairs_df = prepare_pairs(leases)
    segs = two_part_segment_forecast(pairs_df, FORECAST_YEAR, "mean2", "lagged").set_index(["province_regime", "is_renewal"])
    w = expected_segment_leases(leases_clean, pairs_df, FORECAST_YEAR)
    return {"effective_pct": portfolio_figure(segs.pred_effective, w), "contract_pct": portfolio_figure(segs.pred_contract, w)}

scenarios = {
    "bas : concessions au rythme de 2025": estimate_2026(leases_raw, asking_raw, external, "persist"),
    "central : concessions = moyenne 2024-2025": estimate,
    "haut : hausse des concessions divisée par 2": estimate_2026(leases_raw, asking_raw, external, "half"),
    "stabilisation : la part des concessions cesse de monter": estimate_2026(leases_raw, asking_raw, external, "zero"),
    "sensibilité : TAL 2026 inconnu (taux 2025, 5,9 %)": estimate_2026_lagged_regulation(leases_raw),
}
scenario_table = pd.DataFrame({name: {"encaissé %": s["effective_pct"], "au bail %": s["contract_pct"]}
                               for name, s in scenarios.items()}).T.round(2)
print(estimate["definition"])
fr = lambda v: f"{v:.1f}".replace(".", ",")
print(f"\nESTIMATION CENTRALE 2026 : encaissé {fr(estimate['effective_pct'])} %  (au bail {fr(estimate['contract_pct'])} %)")
worst_effective = max(abs(r["effective"]["error"]) for r in backtest_results.values())
print(f"Erreur moyenne historique (MAE {BACKTEST_YEARS[0]}-{BACKTEST_YEARS[-1]}) : {fr(mae_effective)} pt ; "
      f"pire année : {fr(worst_effective)} pt (ce n'est pas un intervalle de confiance)\n")
display(scenario_table)
segment_table = (estimate["segments"][["last_level", "anchor_driver", "spread", "anchor", "pred_contract", "concession_drag", "pred_effective", "expected_leases"]]
 .rename(columns={"last_level": "hausse an dernier", "anchor_driver": "repère 2026", "spread": "écart habituel", "anchor": "repère + écart",
                  "pred_contract": "prévu au bail", "concession_drag": "hausse des concessions", "pred_effective": "prévu encaissé", "expected_leases": "baux attendus"})
 .rename(index={True: "renouvellement", False: "nouveau locataire"}).round(2))
display(segment_table)

segments = estimate["segments"].loc[[("QC_TAL", True), ("QC_TAL", False), ("ON_GUIDELINE", True), ("ON_GUIDELINE", False)]]
segment_names = [f"{'Québec' if r == 'QC_TAL' else 'The Met'}\n{'renewal' if n else 'new tenant'}" for r, n in segments.index]
habit_part = PARTIAL_ADJUSTMENT_WEIGHT * segments.last_level
anchor_part = (1 - PARTIAL_ADJUSTMENT_WEIGHT) * segments.anchor
x = np.arange(len(segments)); width = 0.36
fig, ax = plt.subplots(figsize=(9.6, 5.6))
ax.bar(x - width / 2, habit_part, width, color=CHART_NAVY, label="½ × last year's increase")
ax.bar(x - width / 2, anchor_part, width, bottom=habit_part, color=CHART_GREY, label="½ × (benchmark + usual spread)")
ax.bar(x + width / 2, segments.pred_effective, width, color=CHART_GOLD, label="effective = contract − concessions")
for xi, contract, drag, effective in zip(x, segments.pred_contract, segments.concession_drag, segments.pred_effective):
    ax.text(xi - width / 2, contract + 0.15, f"{contract:.1f}%", ha="center", color=CHART_NAVY)
    ax.text(xi + width / 2, effective + 0.15, f"{effective:.1f}%", ha="center", color=CHART_GOLD, fontweight="bold")
    ax.annotate("", xy=(xi + width / 2, effective + 0.6), xytext=(xi + width / 2, contract),
                arrowprops=dict(arrowstyle="->", color=CHART_GOLD, lw=1.8, linestyle="--"))
    ax.text(xi + width / 2 + 0.05, (contract + effective + 0.6) / 2, f"−{drag:.1f}", fontsize=12, color=CHART_GOLD, va="center")
ax.set_xticks(x, segment_names); pct_axis(ax); ax.set_ylim(0, segments.pred_contract.max() + 1.2)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=12); finish(fig, "fig_equation_2026")

### 7.7 Scénarios 2026 (bas, central, haut)

Hausse encaissée réalisée depuis 2021, prévisions faites au 31 décembre précédent (backtest), puis l'éventail 2026 : **bas** (les concessions montent encore comme en 2025), **central** (rythme moyen 2024–2025), **haut** (leur montée est divisée par 2) et, en pointillé gris, le cas extrême où la part des concessions cesse de monter. Le trait vertical donne l'erreur moyenne passée autour du scénario central.

In [ ]:
# Hausse encaissée réalisée, prévisions faites au 31 décembre précédent (backtest) et éventail des scénarios 2026
SCENARIO_KEYS = {"Low": "bas : concessions au rythme de 2025", "Central": "central : concessions = moyenne 2024-2025",
                 "High": "haut : hausse des concessions divisée par 2",
                 "Concessions stop rising": "stabilisation : la part des concessions cesse de monter"}
SCENARIO_STYLE = {"Low": ("--", CHART_SAND), "Central": ("-", CHART_GOLD), "High": ("--", CHART_SAND), "Concessions stop rising": (":", CHART_GREY)}
history_effective = pd.Series({y: backtest(leases_raw, y)["effective"]["actual"] for y in range(SCORING_START, FORECAST_YEAR)})
past_forecasts = pd.Series({y: backtest_results[y]["effective"]["predicted"] for y in BACKTEST_YEARS})
scenario_values = {label: scenario_table.loc[name, "encaissé %"] for label, name in SCENARIO_KEYS.items()}

last_year, last_value = history_effective.index[-1], history_effective.iloc[-1]
fan_x = [last_year, FORECAST_YEAR]
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.fill_between(fan_x, [last_value, scenario_values["Low"]], [last_value, scenario_values["High"]], color=CHART_SAND, alpha=0.35, lw=0)
ax.plot(history_effective.index, history_effective.values, "-o", color=CHART_NAVY, lw=2.5, ms=7, label="Actual (same unit, effective)")
ax.plot(past_forecasts.index, past_forecasts.values, "o", mfc="none", mec=CHART_GOLD, mew=2, ms=10, label="Forecast made on Dec 31 (backtest)")
for label, value in scenario_values.items():
    style, color = SCENARIO_STYLE[label]
    is_central = label == "Central"
    ax.plot(fan_x, [last_value, value], style, color=color, lw=2.5 if is_central else 1.8)
    ax.plot(FORECAST_YEAR, value, "o", color=color, ms=11 if is_central else 8)
    ax.text(FORECAST_YEAR + 0.12, value, f"{label} {value:.1f}%", va="center", color=CHART_NAVY, fontweight="bold" if is_central else "normal")
ax.errorbar(FORECAST_YEAR - 0.12, scenario_values["Central"], yerr=mae_effective, fmt="none", ecolor=CHART_NAVY, capsize=5, lw=1.5,
            label="± past average error (MAE)")
ax.set_xticks(list(history_effective.index) + [FORECAST_YEAR]); ax.set_xlim(history_effective.index[0] - 0.3, FORECAST_YEAR + 1.7)
ax.set_ylim(0, max(scenario_values.values()) + 1.5); pct_axis(ax); ax.legend(loc="upper left", fontsize=11)
finish(fig, "fig_scenarios")

### 7.8 Sensibilité : fenêtre de l'écart habituel

L'écart habituel au repère est moyenné sur `SPREAD_WINDOW_YEARS` = 3 ans, valeur **fixée avant de comparer** (1 an est trop bruité ; une fenêtre longue mélange la période où les nouveaux locataires payaient bien moins que le marché, 2022–2024, et le rattrapage de 2025). On montre ici ce que donneraient d'autres fenêtres, sans changer le choix : retenir après coup la fenêtre au meilleur backtest serait du surapprentissage.

In [ ]:
SPREAD_WINDOWS_TESTED = {"1 year": 1, "2 years": 2, "3 years (retained)": 3, "4 years": 4, "all years": SPREAD_SEARCH_YEARS}
retained_window = SPREAD_WINDOW_YEARS
sensitivity_rows = {}
for label, window in SPREAD_WINDOWS_TESTED.items():
    SPREAD_WINDOW_YEARS = window                  # lu par two_part_segment_forecast
    estimate_w = estimate_2026(leases_raw, asking_raw, external)
    errors_required = [backtest(leases_raw, y)["effective"]["error"] for y in BACKTEST_YEARS]
    errors_long = [backtest(leases_raw, y)["effective"]["error"] for y in range(SCORING_START, FORECAST_YEAR)]
    sensitivity_rows[label] = {f"{FORECAST_YEAR} encaissé %": estimate_w["effective_pct"],
                               f"{FORECAST_YEAR} au bail %": estimate_w["contract_pct"],
                               f"MAE {BACKTEST_YEARS[0]}-{LAST_OBSERVED_YEAR}": np.mean(np.abs(errors_required)),
                               f"MAE {SCORING_START}-{LAST_OBSERVED_YEAR}": np.mean(np.abs(errors_long))}
SPREAD_WINDOW_YEARS = retained_window
assert abs(estimate_2026(leases_raw, asking_raw, external)["effective_pct"] - estimate["effective_pct"]) < 1e-9
sensitivity_table = pd.DataFrame(sensitivity_rows).T
display(sensitivity_table.round(2))

low, high = scenario_table["encaissé %"].iloc[0], scenario_table["encaissé %"].iloc[2]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axhspan(low, high, color=CHART_LIGHT, label="scenario range (low – high)")
ax.plot(sensitivity_table.index, sensitivity_table[f"{FORECAST_YEAR} encaissé %"], "-o", color=CHART_NAVY, lw=2, ms=8)
ax.plot(["3 years (retained)"], [estimate["effective_pct"]], "o", color=CHART_GOLD, ms=13, label="retained")
ax.set_ylim(low - 0.6, high + 0.6); pct_axis(ax); ax.set_title("2026 effective increase by usual-spread window"); ax.legend()
finish(fig)

**Lecture.** Selon la fenêtre, l'estimation 2026 va de 3,2 % à 4,1 % encaissé, dans la plage des scénarios (2,4 % – 4,1 %). Les fenêtres longues font un peu mieux sur le passé, mais tous les écarts restent sous le seuil d'égalité de 0,3 pt : le choix de 3 ans est maintenu, et 3,2 % se lit comme une estimation basse-centrale.

### 7.9 Hypothèses

- Année d'une hausse = année de début du bail ; paires entre 0,75 et 2,5 ans d'écart.
- Immeubles québécois non chauffés (taux du TAL « non chauffé ») ; The Met exempté de la ligne directrice ontarienne (premier bail en 2023, à confirmer avec Équinoxe).
- Information : données publiées au 31 décembre de l'année précédente, sauf le taux du TAL de l'année visée, publié en janvier avant les avis de renouvellement (sans lui, 2026 donne 4,4 %).
- Part de renouvellements 2026 = celle de 2025 ; baux attendus = baux en cours qui finissent d'ici fin 2026 (pondération seulement, l'extrait ne donne pas de volumes réels).
- Écart habituel au repère moyenné sur 3 ans et poids ½ fixés d'avance ; hausse des concessions = moyenne des 2 dernières années (scénario central).
- Les loyers affichés (`sAskingRent`) ont été testés comme signal avancé (écart affiché / signé stable à ~2 %) et écartés.

### 7.10 Fichier du modèle

Paramètres agrégés par segment (aucune ligne du CRM) dans `models/two_part_2026.json`.

In [ ]:
seg = estimate["segments"]
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
model_record = {
    "method": "modèle en deux parties : hausse au bail = w × hausse de l'an dernier + (1 − w) × (repère + écart habituel) ; hausse encaissée = hausse au bail − hausse attendue des concessions",
    "partial_adjustment_weight": PARTIAL_ADJUSTMENT_WEIGHT,
    "spread_window_years": SPREAD_WINDOW_YEARS,
    "concession_rule_central": "mean2",
    "selection": "à égalité statistique avec la référence (même hausse que l'an dernier) ; retenu car il intègre l'information connue de 2026 (TAL, Ontario, SCHL) — section 7",
    "estimate_2026": {"effective_pct": round(estimate["effective_pct"], 3), "contract_pct": round(estimate["contract_pct"], 3)},
    "scenarios": scenario_table.to_dict(orient="index"),
    "backtest_mae_2023_2025": {"effective": round(mae_effective, 3), "contract": round(mae_contract, 3)},
    "backtest_worst_abs_error_2023_2025": {"effective": round(max(abs(r["effective"]["error"]) for r in backtest_results.values()), 2),
                                           "contract": round(max(abs(r["contract"]["error"]) for r in backtest_results.values()), 2)},
    "information_timing": "données publiées au 31 déc. de l'année précédente, sauf le taux du TAL de l'année visée (publié en janvier, connu avant les avis de renouvellement)",
    "segments": [{**{k: (bool(v) if isinstance(v, (bool, np.bool_)) else v) for k, v in zip(["province_regime", "is_renewal"], idx)},
                  **{c: round(float(seg.loc[idx, c]), 3) for c in ["last_level", "anchor_driver", "spread", "anchor",
                                                                   "pred_contract", "concession_drag", "pred_effective", "expected_leases"]}}
                 for idx in seg.index],
}
(MODEL_DIR / "two_part_2026.json").write_text(json.dumps(model_record, indent=2, ensure_ascii=False))
print(json.dumps(model_record["estimate_2026"]), "->", MODEL_DIR / "two_part_2026.json")

# Le fichier suffit à refaire l'estimation : moyenne des hausses encaissées des 4 groupes, pondérée par les baux attendus
reloaded = json.loads((MODEL_DIR / "two_part_2026.json").read_text())
reloaded_segments = pd.DataFrame(reloaded["segments"])
from_file = np.average(reloaded_segments.pred_effective, weights=reloaded_segments.expected_leases)
print(f"Estimation recalculée depuis le fichier : {from_file:.2f} % (notebook : {estimate['effective_pct']:.2f} %)".replace(".", ","))
assert abs(from_file - estimate["effective_pct"]) < 0.01

### 7.11 Vérification après coup (données 2026, jamais utilisées en entrée)

In [ ]:
CUTOFF = pd.Timestamp("2025-12-31")
post = cpi_monthly[cpi_monthly.series.isin(["rent_quebec", "rent_ontario"])].copy()
post["yoy_pct"] = post.groupby("series").index_value.pct_change(MONTHS_PER_YEAR) * 100
recent = post[post.month >= f"{LAST_OBSERVED_YEAR}-01-01"].pivot(index="month", columns="series", values="yoy_pct").round(2)
recent.index = recent.index.strftime("%Y-%m")
after = post[post.month > CUTOFF].groupby("series").yoy_pct.mean()
before = post[post.month.dt.year == LAST_OBSERVED_YEAR].groupby("series").yoy_pct.mean()
print("IPC loyers, variation annuelle moyenne : 2025 ->", before.round(2).to_dict(), "| 2026 à date ->", after.round(2).to_dict())

fig, ax = plt.subplots(figsize=(9, 3.6))
for series, color, label in [("rent_quebec", CHART_NAVY, "Québec"), ("rent_ontario", CHART_GOLD, "Ontario")]:
    ax.plot(pd.to_datetime(recent.index), recent[series], "-o", color=color, ms=4, label=f"CPI rent {label}")
ax.axvline(CUTOFF, color=CHART_GREY, ls="--"); ax.text(CUTOFF, ax.get_ylim()[1], " forecast date", va="top", color=CHART_GREY)
ax.set_ylabel("12-month change"); pct_axis(ax); ax.legend(); finish(fig)

## 8. Bonus : prévision par immeuble et par nombre de chambres

La hausse étant décidée pour tout le portefeuille, un immeuble ou un type de logement ne diffère que par son régime (Québec / Ontario) et sa part de renouvellements. `predictions_2026.csv` : une ligne par immeuble × chambres (0 = studio) × statut, avec la hausse au bail et la hausse encaissée, et le nombre de baux attendus dans l'extrait (pondération seulement). Les lignes « Tous » agrègent ; la ligne Portefeuille reprend l'estimation officielle.

Le tableau de bord par immeuble, avec choix du scénario, est produit en annexe C.

In [ ]:
leases_clean_all, pairs_all = prepare_pairs(leases_raw)
current_leases = leases_clean_all.sort_values("lease_start").groupby(["prop_code", "unit_code"]).tail(1)
due_2026 = current_leases[current_leases.lease_end.dt.year <= FORECAST_YEAR]
renewal_share_by_building = pairs_all[pairs_all.lease_year == LAST_OBSERVED_YEAR].groupby("building").is_renewal.mean()

PROVINCE_OF_REGIME = {"QC_TAL": "Québec", "ON_GUIDELINE": "Ontario"}
STATUS_LABEL = {True: "renouvellement", False: "nouveau locataire"}
DISPLAY_NAMES = {"Levesque": "Lévesque", "Saint-Elzear": "Saint-Elzéar"}
prediction_rows = []
for (building, beds), g in due_2026.groupby(["building", "beds"]):
    regime = g.province_regime.iloc[0]
    share = renewal_share_by_building.get(building)
    for renewal in (True, False):
        prediction_rows.append({"immeuble": building, "province": PROVINCE_OF_REGIME[regime], "chambres": int(beds),
                                "statut": STATUS_LABEL[renewal],
                                "baux_attendus_extrait": len(g) * (share if renewal else 1 - share),
                                "hausse_au_bail_pct": seg.loc[(regime, renewal), "pred_contract"],
                                "hausse_encaissee_pct": seg.loc[(regime, renewal), "pred_effective"]})
predictions = pd.DataFrame(prediction_rows)

def weighted_rows(keys, labels):
    out = []
    for key, g in predictions.groupby(keys):
        key = key if isinstance(key, tuple) else (key,)
        w = g.baux_attendus_extrait
        out.append({**labels, **dict(zip(keys, key)), "baux_attendus_extrait": w.sum(),
                    "hausse_au_bail_pct": np.average(g.hausse_au_bail_pct, weights=w),
                    "hausse_encaissee_pct": np.average(g.hausse_encaissee_pct, weights=w)})
    return pd.DataFrame(out)

summary_rows = pd.concat([
    weighted_rows(["immeuble", "province"], {"chambres": "Tous", "statut": "Tous"}),
    weighted_rows(["chambres"], {"immeuble": "Tous", "province": "Tous", "statut": "Tous"}),
    weighted_rows(["statut"], {"immeuble": "Tous", "province": "Tous", "chambres": "Tous"}),
    pd.DataFrame([{"immeuble": "Portefeuille", "province": "Tous", "chambres": "Tous", "statut": "Tous",
                   "baux_attendus_extrait": predictions.baux_attendus_extrait.sum(),
                   "hausse_au_bail_pct": estimate["contract_pct"], "hausse_encaissee_pct": estimate["effective_pct"]}]),
])
predictions_2026 = pd.concat([predictions, summary_rows], ignore_index=True)[
    ["immeuble", "province", "chambres", "statut", "baux_attendus_extrait", "hausse_au_bail_pct", "hausse_encaissee_pct"]].round(2)
predictions_2026.to_csv("predictions_2026.csv", index=False)
print(f"predictions_2026.csv : {len(predictions)} lignes détaillées + {len(summary_rows)} lignes agrégées")
display(predictions_2026[predictions_2026.chambres.eq("Tous") | predictions_2026.immeuble.eq("Tous")])

by_building = (predictions_2026[predictions_2026.chambres.eq("Tous") & predictions_2026.statut.eq("Tous")
                                & ~predictions_2026.immeuble.isin(["Tous", "Portefeuille"])].sort_values("hausse_encaissee_pct"))
names = by_building.immeuble.replace(DISPLAY_NAMES) + by_building.province.map({"Québec": "", "Ontario": " (Ontario)"})
y = np.arange(len(by_building)); height = 0.38
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.barh(y + height / 2, by_building.hausse_au_bail_pct, height, color=CHART_NAVY, label="Contract")
ax.barh(y - height / 2, by_building.hausse_encaissee_pct, height, color=CHART_GOLD, label="Effective")
for yi, v in zip(y + height / 2, by_building.hausse_au_bail_pct): ax.text(v + 0.08, yi, f"{v:.1f}%", va="center", color=CHART_NAVY)
for yi, v in zip(y - height / 2, by_building.hausse_encaissee_pct): ax.text(v + 0.08, yi, f"{v:.1f}%", va="center", color=CHART_GOLD)
ax.set_yticks(y, names); pct_axis(ax, "x"); ax.set_xlim(0, by_building.hausse_au_bail_pct.max() + 1.5)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2); finish(fig, "fig_immeubles")

## Annexes

Analyses non retenues pour la prévision, gardées pour justifier le choix d'un modèle simple : les caractéristiques des logements expliquent le **prix**, pas la **hausse**.

### A. Explicabilité : rôle de chaque variable

Part de la croissance expliquée (R²), valeurs SHAP (XGBoost) et importance par permutation, ACP des caractéristiques des logements.

In [ ]:
# Explicabilité : R², SHAP, permutation, ACP
from sklearn.decomposition import PCA

EXPLAIN_YEARS = range(2021, 2026)
ex = pairs[pairs.lease_year.isin(EXPLAIN_YEARS)].copy()
ex["prev_lease_id"] = ex.lease_id - 1
ex["prev_concession_share"] = ex.prev_lease_id.map(prev_concession_share).fillna(0)
ex["log_prev_rent"] = np.log(ex.prev_rent_contract)
ex["log_sqft"] = np.log(ex.sqft)
ex["rent_per_sqft"] = ex.prev_rent_contract / ex.sqft
ex["year"] = ex.lease_year.astype(str)

# Q1 incremental R^2
FACTOR_SETS = [
    ("année", "C(year)"),
    ("année × renouvellement", "C(year) * is_renewal"),
    ("+ immeuble", "C(year) * is_renewal + C(prop_code)"),
    ("+ chambres, salles de bain, superficie, étage", "C(year) * is_renewal + C(prop_code) + C(beds) + baths + log_sqft + floor"),
    ("+ loyer précédent, concessions précédentes, rang du bail", "C(year) * is_renewal + C(prop_code) + C(beds) + baths + log_sqft + floor + log_prev_rent + prev_concession_share + lease_seq"),
    ("+ interactions année × province, immeuble, chambres", "C(year) * is_renewal * province_regime + C(prop_code) * C(year) + C(beds) * C(year) + log_prev_rent + prev_concession_share"),
]
import warnings
r2_rows = []
for label, rhs in FACTOR_SETS:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # interaction sets are rank-deficient by design; R² is unaffected
        r2_rows.append({"facteurs": label,
                        "R² hausse au bail": smf.ols("growth_rent_contract ~ " + rhs, ex).fit().rsquared,
                        "R² hausse encaissée": smf.ols("growth_rent_effective_crm ~ " + rhs, ex).fit().rsquared})
r2_table = pd.DataFrame(r2_rows).set_index("facteurs")

# Q2 XGBoost on all features, permutation importance on the last year, SHAP on all
UNIT_FEATURES = ["renewal", "is_ontario", "beds", "baths", "log_sqft", "floor", "log_prev_rent",
                 "prev_concession_share", "lease_seq", "rent_per_sqft"]
ex["renewal"] = ex.is_renewal.astype(float); ex["is_ontario"] = (ex.province_regime == "ON_GUIDELINE").astype(float)
year_dummies = pd.get_dummies(ex.lease_year, prefix="year", dtype=float)
building_dummies = pd.get_dummies(ex.prop_code, prefix="bldg", dtype=float)
X_ex = pd.concat([ex[UNIT_FEATURES], year_dummies, building_dummies], axis=1)
PERMUTATION_REPEATS = 5
importance = {}
for target in ["growth_rent_contract", "growth_rent_effective_crm"]:
    model = xgb.XGBRegressor(**XGB_PARAMS).fit(X_ex, ex[target])          # explanatory fit on all years
    # SHAP values computed by XGBoost itself (TreeSHAP, pred_contribs); the last column is the bias term
    shap_values = model.get_booster().predict(xgb.DMatrix(X_ex), pred_contribs=True)[:, :-1]
    mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=X_ex.columns)
    groups = {"année (décision commune)": [c for c in X_ex if c.startswith("year_")],
              "immeuble": [c for c in X_ex if c.startswith("bldg_")]}
    grouped = {g: mean_abs[cols].sum() for g, cols in groups.items()}
    grouped.update({c: mean_abs[c] for c in UNIT_FEATURES})
    # grouped permutation importance (in-sample, explanatory): shuffle a whole group of columns together
    rng = np.random.default_rng(0)
    base_r2 = model.score(X_ex, ex[target])
    perm_grouped = {}
    for name, cols in {**groups, **{c: [c] for c in UNIT_FEATURES}}.items():
        drops = []
        for _ in range(PERMUTATION_REPEATS):
            Xp = X_ex.copy(); Xp[cols] = Xp[cols].values[rng.permutation(len(Xp))]
            drops.append(base_r2 - model.score(Xp, ex[target]))
        perm_grouped[name] = np.mean(drops)
    importance[target] = pd.DataFrame({"|SHAP| moyen (pts)": pd.Series(grouped), "importance par permutation (perte de R²)": pd.Series(perm_grouped)})
importance_table = pd.concat(importance, axis=1).sort_values(("growth_rent_contract", "|SHAP| moyen (pts)"), ascending=False)

# Q3 PCA on unit attributes
PCA_FEATURES = ["beds", "baths", "log_sqft", "floor", "log_prev_rent", "rent_per_sqft", "latitude", "longitude"]
units_now = ex.sort_values("lease_start").groupby(["prop_code", "unit_code"]).tail(1)
Z = StandardScaler().fit_transform(units_now[PCA_FEATURES])
pca = PCA(n_components=4).fit(Z)
loadings = pd.DataFrame(pca.components_.T, index=PCA_FEATURES, columns=[f"PC{i+1}" for i in range(4)])
explained = pd.Series(pca.explained_variance_ratio_, index=loadings.columns)
scores = pd.DataFrame(pca.transform(StandardScaler().fit(units_now[PCA_FEATURES]).transform(ex[PCA_FEATURES])),
                      columns=loadings.columns, index=ex.index)
ex["growth_deviation"] = ex.growth_rent_contract - ex.groupby(["lease_year", "is_renewal", "province_regime"]).growth_rent_contract.transform("median")
pc_vs_growth = pd.DataFrame({"corr. hausse au bail": scores.corrwith(ex.growth_rent_contract),
                             "corr. écart au segment": scores.corrwith(ex.growth_deviation),
                             "corr. loyer au pi²": scores.corrwith(ex.rent_per_sqft)})

display(r2_table.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, target, title in zip(axes, ["growth_rent_contract", "growth_rent_effective_crm"], ["contract increase", "effective increase"]):
    importance_table[(target, "|SHAP| moyen (pts)")].sort_values().plot.barh(ax=ax, color=CHART_NAVY)
    ax.set_title(f"mean |SHAP| — {title} (pts)"); ax.grid(axis="x", alpha=0.3)
finish(fig)

In [ ]:
print("Variance expliquée :", explained.round(3).to_dict())
display(loadings.round(2))
display(pc_vs_growth.round(3))
fig, ax = plt.subplots(figsize=(7, 5.5))
unit_scores = pd.DataFrame(pca.transform(Z)[:, :2], columns=["PC1", "PC2"], index=units_now.index)
for building, g in units_now.groupby("building"):
    ax.scatter(unit_scores.loc[g.index, "PC1"], unit_scores.loc[g.index, "PC2"], s=8, alpha=0.6, label=building)
ax.set_xlabel(f"PC1 ({explained['PC1']:.0%}) : location and price per sq ft")
ax.set_ylabel(f"PC2 ({explained['PC2']:.0%}) : size and rent level")
ax.legend(fontsize=7, markerscale=2); ax.grid(alpha=0.3); ax.set_title("Units by characteristics")
finish(fig)

**Lecture.** La hausse au bail = l'année (décision commune) + le type de bail ; immeuble, chambres, superficie et étage ajoutent < 0,1 pt de R². La hausse encaissée d'un logement dépend surtout des concessions de son bail précédent (R² de 8 % à 35 %). L'ACP : deux axes (emplacement / prix au pi², taille / loyer) résument ~74 % des différences entre logements, sans lien avec leur hausse (corrélation ≈ 0,00). Les caractéristiques fixent le **prix**, pas la **hausse**.

### B. Modèle à deux niveaux (écarts par logement, hyperparamètres par province)

In [ ]:
# Modèle à deux niveaux : segment (deux parties) + écarts par logement, hyperparamètres par province
DEV_FEATURES = ["renewal", "beds", "baths", "log_sqft", "floor", "log_prev_rent", "prev_concession_share", "lease_seq", "rent_per_sqft"]
HYPER_GRID = [dict(max_depth=d, min_child_weight=m) for d in (2, 3, 4) for m in (10, 30)]
DEV_BASE = dict(n_estimators=200, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=0)
TWO_LEVEL_TEST_YEARS = [2023, 2024, 2025]

for target in ["growth_rent_contract", "growth_rent_effective_crm"]:
    ex[f"dev_{target}"] = ex[target] - ex.groupby(["lease_year", "is_renewal", "province_regime"])[target].transform("median")

def design_dev(d):
    return pd.concat([d[DEV_FEATURES], pd.get_dummies(d.prop_code, prefix="bldg", dtype=float)], axis=1)

def tune_and_fit(train_df, target):
    # choose hyperparameters on the last training year (inner rolling validation), then refit on all training years
    years = sorted(train_df.lease_year.unique())
    if len(years) < 2:
        params = HYPER_GRID[0]
    else:
        inner_tr, inner_va = train_df[train_df.lease_year < years[-1]], train_df[train_df.lease_year == years[-1]]
        X_tr = design_dev(inner_tr); X_va = design_dev(inner_va).reindex(columns=X_tr.columns, fill_value=0)
        scores = []
        for hp in HYPER_GRID:
            m = xgb.XGBRegressor(**DEV_BASE, **hp).fit(X_tr, inner_tr[f"dev_{target}"])
            scores.append(np.abs(m.predict(X_va) - inner_va[f"dev_{target}"]).mean())
        params = HYPER_GRID[int(np.argmin(scores))]
    X = design_dev(train_df)
    return xgb.XGBRegressor(**DEV_BASE, **params).fit(X, train_df[f"dev_{target}"]), params, list(X.columns)

two_level_rows, chosen_hyper = [], {}
for target in ["growth_rent_contract", "growth_rent_effective_crm"]:
    for y in TWO_LEVEL_TEST_YEARS:
        for regime in ["QC_TAL", "ON_GUIDELINE"]:
            tr = ex[(ex.lease_year < y) & (ex.province_regime == regime)]
            te = ex[(ex.lease_year == y) & (ex.province_regime == regime)]
            if len(tr) == 0 or len(te) == 0: continue
            model, hp, cols = tune_and_fit(tr, target)
            chosen_hyper[(target, y, regime)] = hp
            dev_pred = model.predict(design_dev(te).reindex(columns=cols, fill_value=0))
            seg_med = te[target] - te[f"dev_{target}"]               # realised segment median (isolates the unit-level skill)
            two_level_rows.append(pd.DataFrame({"target": target, "year": y, "regime": regime, "building": te.building,
                                                "beds": te.beds, "actual": te[target], "segment_only": seg_med,
                                                "two_level": seg_med + dev_pred}))
two_level = pd.concat(two_level_rows)

def skill(df, level):
    if level == "unit":
        g = df
    else:
        g = df.groupby(["target", "year", level])[["actual", "segment_only", "two_level"]].median().reset_index()
    return g.groupby("target").apply(lambda d: pd.Series({
        "MAE segment seul": (d.segment_only - d.actual).abs().mean(),
        "MAE deux niveaux": (d.two_level - d.actual).abs().mean()}), include_groups=False)
two_level_skill = pd.concat({lvl: skill(two_level, lvl) for lvl in ["unit", "building", "beds"]}, names=["level"])
two_level_skill["gain (pts)"] = two_level_skill["MAE segment seul"] - two_level_skill["MAE deux niveaux"]

display(pd.Series({f"{t.replace('growth_rent_', '')} {y} {r}": f"profondeur {hp['max_depth']}, min_child_weight {hp['min_child_weight']}"
           for (t, y, r), hp in chosen_hyper.items()}, name="hyperparamètres choisis par province").to_frame())

skill_plot = two_level_skill.reset_index()
skill_plot["label"] = skill_plot.level.map({"unit": "unit", "building": "building", "beds": "bedrooms"}) + "\n" + \
    skill_plot.target.str.replace("growth_rent_contract", "contract").str.replace("growth_rent_effective_crm", "effective")
fig, ax = plt.subplots(figsize=(14, 5)); x = np.arange(len(skill_plot)); width = 0.38
ax.bar(x - width / 2, skill_plot["MAE segment seul"], width, color=CHART_NAVY, label="segment only")
ax.bar(x + width / 2, skill_plot["MAE deux niveaux"], width, color=CHART_SAND, label="two-level")
ax.set_xticks(x, skill_plot.label, fontsize=9); ax.set_ylabel("mean error (pts)"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("The two-level model barely improves anything"); finish(fig)

Gain faible sur la hausse au bail par immeuble ou type de logement, perte nette sur la hausse encaissée : **non retenu**.

### C. Tableau de bord par immeuble

`dashboard_2026.html` : page autonome (à ouvrir dans un navigateur, hors ligne) avec, pour chaque immeuble et pour le portefeuille, la hausse 2026 au bail et encaissée, l'historique même logement, les baux attendus par type de logement et l'écart au marché (Kaggle). Des boutons permettent de **choisir le scénario de concessions** (bas, central, haut, concessions stables) : tous les chiffres se mettent à jour. Agrégats seulement, aucune ligne du CRM.

In [ ]:
# Tableau de bord HTML autonome, avec choix du scénario de concessions (agrégats seulement, aucune ligne du CRM)
DASHBOARD_FILE = Path("dashboard_2026.html")
MIN_PAIRS_PER_POINT = 10      # point historique masqué s'il repose sur moins de paires même logement
BED_LABELS = {0: "Studio", 1: "1 chambre", 2: "2 chambres", 3: "3 chambres"}
DASHBOARD_SCENARIOS = {"persist": ("Bas", "les concessions montent encore comme en 2025"),
                       "mean2": ("Central", "les concessions montent au rythme moyen 2024–2025"),
                       "half": ("Haut", "la montée des concessions est divisée par 2"),
                       "zero": ("Concessions stables", "la part des concessions cesse de monter")}
REGIME_OF_PROVINCE = {province: regime for regime, province in PROVINCE_OF_REGIME.items()}
scenario_estimates = {rule: estimate_2026(leases_raw, asking_raw, external, rule) for rule in DASHBOARD_SCENARIOS}
history_years = list(range(DISPLAY_START_YEAR, FORECAST_YEAR))

def scenario_values(detail, measure):
    """Hausse moyenne des lignes détaillées (immeuble × chambres × statut), pondérée par les baux attendus, pour chaque scénario."""
    keys = list(zip(detail.province.map(REGIME_OF_PROVINCE), detail.statut.eq("renouvellement")))
    return {rule: round(float(np.average([est["segments"].loc[k, measure] for k in keys], weights=detail.baux_attendus_extrait)), 2)
            for rule, est in scenario_estimates.items()}

def history_points(pairs_df):
    g = pairs_df[pairs_df.lease_year.isin(history_years)].groupby("lease_year")
    h = pd.DataFrame({"contract": g.growth_rent_contract.median(), "effective": g.growth_rent_effective_crm.median(), "pairs": g.size()})
    h = h[h.pairs >= MIN_PAIRS_PER_POINT].round(2)
    return [{"year": int(y), "contract": float(r.contract), "effective": float(r.effective)} for y, r in h.iterrows()]

def mix_rows(detail):
    totals = detail.groupby(["chambres", "statut"]).baux_attendus_extrait.sum().unstack()
    return [{"label": BED_LABELS.get(int(beds), f"{beds} chambres"), "renewal": round(float(row["renouvellement"]), 1),
             "new": round(float(row["nouveau locataire"]), 1)} for beds, row in totals.iterrows()]

gap_by_building = market_gap["écart au marché, juin 2024 (%)"].dropna() if "market_gap" in globals() else pd.Series(dtype=float)
entries = []
for name, detail in [("Portefeuille", predictions)] + list(predictions.groupby("immeuble")):
    is_portfolio = name == "Portefeuille"
    province = "Québec et Ontario" if is_portfolio else detail.province.iloc[0]
    effective = ({rule: round(est["effective_pct"], 2) for rule, est in scenario_estimates.items()} if is_portfolio
                 else scenario_values(detail, "pred_effective"))
    contract = round(estimate["contract_pct"], 2) if is_portfolio else scenario_values(detail, "pred_contract")["mean2"]
    entries.append({"id": name, "name": DISPLAY_NAMES.get(name, name), "province": province,
                    "rule": {"Québec": "TAL : taux de référence 2026", "Ontario": "exempté de la ligne directrice (premiers baux en 2023)"}.get(
                        province, "TAL (Québec) et ligne directrice (Ontario)"),
                    "contract": contract, "effective": effective, "leases": float(detail.baux_attendus_extrait.sum()),
                    "renewal_share": float(detail[detail.statut.eq("renouvellement")].baux_attendus_extrait.sum() / detail.baux_attendus_extrait.sum()),
                    "market_gap": None if is_portfolio or name not in gap_by_building else round(float(gap_by_building[name]), 1),
                    "history": history_points(pairs_all if is_portfolio else pairs_all[pairs_all.building.eq(name)]),
                    "mix": mix_rows(detail)})
segment_rates = {rule: {f"{PROVINCE_OF_REGIME[r]}|{STATUS_LABEL[bool(n)]}": round(float(est["segments"].loc[(r, n), "pred_effective"]), 2)
                        for r, n in est["segments"].index} for rule, est in scenario_estimates.items()}
dashboard_data = {"year": FORECAST_YEAR, "last_year": LAST_OBSERVED_YEAR, "first_year": DISPLAY_START_YEAR, "mae": round(float(mae_effective), 2),
                  "min_pairs": MIN_PAIRS_PER_POINT, "segments": segment_rates, "entries": entries,
                  "scenarios": [{"id": rule, "label": label, "note": note} for rule, (label, note) in DASHBOARD_SCENARIOS.items()]}
DASHBOARD_TEMPLATE = r"""<!doctype html><html lang="fr"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Prévision 2026 par immeuble</title>
<link href="https://fonts.googleapis.com/css2?family=Lato:wght@400;700&family=Lora:wght@500;600&display=swap" rel="stylesheet">
<style>
:root{--navy:#0B1F40;--gold:#8B7543;--sand:#B9AB88;--grey:#8A93A0;--line:#E3E6EC;--paper:#F1F3F7;--white:#fff}
*{box-sizing:border-box}body{margin:0;background:var(--paper);color:var(--navy);font:15px/1.45 Lato,system-ui,sans-serif}
.wrap{max-width:1180px;margin:0 auto;padding:28px 20px 40px}
h1{font:600 28px/1.2 Lora,Georgia,serif;margin:0}h2{font:600 17px Lora,Georgia,serif;margin:0 0 12px}
.sub{color:var(--grey);margin:6px 0 18px}.rule{height:3px;width:56px;background:var(--gold);margin:14px 0 6px}
.row{display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin-bottom:14px}.row .lbl{font-size:12px;letter-spacing:.06em;text-transform:uppercase;color:var(--grey);margin-right:6px}
.row button{font:700 14px Lato,sans-serif;border:1px solid var(--line);background:var(--white);color:var(--navy);padding:8px 14px;border-radius:999px;cursor:pointer}
.row button[aria-pressed=true]{background:var(--navy);color:#fff;border-color:var(--navy)}
.row.scen button[aria-pressed=true]{background:var(--gold);border-color:var(--gold)}
.note-s{color:var(--grey);font-size:13px;margin:-6px 0 18px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;margin-bottom:16px}
.kpi{background:var(--white);border-radius:10px;padding:14px 16px;border-top:3px solid var(--line)}
.kpi.main{border-top-color:var(--gold)}.kpi .l{font-size:12px;letter-spacing:.06em;text-transform:uppercase;color:var(--grey)}
.kpi .v{font:600 30px Lora,Georgia,serif;margin-top:4px}.kpi .n{font-size:13px;color:var(--grey)}
.grid{display:grid;grid-template-columns:1.25fr 1fr;gap:16px}.card{background:var(--white);border-radius:10px;padding:18px}
.full{grid-column:1/-1}.legend{display:flex;gap:16px;font-size:13px;color:var(--grey);margin:-4px 0 6px;flex-wrap:wrap}
.sw{display:inline-block;width:12px;height:12px;border-radius:2px;margin-right:6px;vertical-align:-1px}
svg{width:100%;height:auto;display:block}svg text{font-family:Lato,sans-serif;fill:var(--navy)}
.note{font-size:13px;color:var(--grey);margin-top:18px}
@media(max-width:820px){.grid{grid-template-columns:1fr}}
</style></head><body><div class="wrap">
<h1>Collection Équinoxe — hausse de loyer <span id="yr"></span> par immeuble</h1><div class="rule"></div>
<p class="sub">Hausse du loyer d'un même logement par rapport à son bail précédent. Encaissé = loyer au bail moins les concessions.</p>
<div class="row" id="tabs"><span class="lbl">Immeuble</span></div>
<div class="row scen" id="scens"><span class="lbl">Scénario</span></div><p class="note-s" id="scen-note"></p>
<div class="kpis" id="kpis"></div>
<div class="grid">
 <div class="card"><h2>Historique et prévision</h2><div class="legend"><span><i class="sw" style="background:var(--navy)"></i>au bail</span><span><i class="sw" style="background:var(--gold)"></i>encaissé</span><span>pointillés : prévision</span></div><div id="hist"></div></div>
 <div class="card"><h2>Baux attendus par type de logement</h2><div class="legend"><span><i class="sw" style="background:var(--navy)"></i>renouvellement</span><span><i class="sw" style="background:var(--sand)"></i>nouveau locataire</span></div><div id="mix"></div></div>
 <div class="card full"><h2>Comparaison des immeubles (encaissé, scénario choisi)</h2><div id="cmp"></div></div>
</div><p class="note" id="note"></p></div>
<script>
const D=__DATA__;const NS="http://www.w3.org/2000/svg";
const fr=(v,d=1)=>v.toFixed(d).replace(".",",");
function el(t,a,p,txt){const e=document.createElementNS(NS,t);for(const k in a)e.setAttribute(k,a[k]);if(txt!==undefined)e.textContent=txt;if(p)p.appendChild(e);return e}
function svg(w,h){return el("svg",{viewBox:`0 0 ${w} ${h}`,role:"img"})}
const PF=D.entries.find(e=>e.id==="Portefeuille");let cur=PF,scen="mean2";
const eff=e=>e.effective[scen];
document.getElementById("yr").textContent=D.year;
function buttons(id,items,isOn,onPick){const box=document.getElementById(id);[...box.querySelectorAll("button")].forEach(b=>b.remove());
 items.forEach(it=>{const b=document.createElement("button");b.textContent=it.label;b.setAttribute("aria-pressed",isOn(it));b.onclick=()=>{onPick(it);render()};box.appendChild(b)})}
function kpis(){const e=cur,k=document.getElementById("kpis"),sc=D.scenarios.find(s=>s.id===scen);
 const items=[["main","Hausse encaissée "+D.year,fr(eff(e))+" %",e===PF?"scénario "+sc.label.toLowerCase():"portefeuille "+fr(eff(PF))+" %"],
  ["","Hausse au bail "+D.year,fr(e.contract)+" %","même dans tous les scénarios"],
  ["","Baux attendus (extrait)",Math.round(e.leases),`${Math.round(e.renewal_share*100)} % de renouvellements`],
  ["","Régime",e.province,e.rule]];
 if(e.market_gap!==null)items.push(["","Écart au marché (Kaggle)",(e.market_gap>0?"+":"")+fr(e.market_gap)+" %","loyer affiché vs comparables, juin 2024"]);
 if(e===PF)items.push(["","Erreur moyenne passée","±"+fr(D.mae)+" pt","MAE des prévisions 2023–"+D.last_year]);
 k.innerHTML=items.map(([c,l,v,n])=>`<div class="kpi ${c}"><div class="l">${l}</div><div class="v" style="${v.length>8?'font-size:20px':''}">${v}</div><div class="n">${n}</div></div>`).join("")}
function hist(){const box=document.getElementById("hist");box.innerHTML="";const W=620,H=300,m={l:44,r:56,t:16,b:34};
 const pts=cur.history,x0=D.first_year,x1=D.year,years=Array.from({length:x1-x0+1},(_,i)=>x0+i),fc={contract:cur.contract,effective:eff(cur)};
 const vals=[...pts.flatMap(p=>[p.contract,p.effective]),fc.contract,fc.effective,0],lo=Math.floor(Math.min(...vals)),hi=Math.ceil(Math.max(...vals))+1;
 const X=y=>m.l+(y-x0)/(x1-x0)*(W-m.l-m.r),Y=v=>H-m.b-(v-lo)/(hi-lo)*(H-m.t-m.b);const s=svg(W,H);box.appendChild(s);
 el("rect",{x:X(x1-0.5),y:m.t,width:X(x1)-X(x1-0.5)+28,height:H-m.t-m.b,fill:"#F1F3F7"},s);
 for(let v=lo;v<=hi;v+=Math.max(1,Math.round((hi-lo)/5))){el("line",{x1:m.l,x2:W-m.r,y1:Y(v),y2:Y(v),stroke:v===0?"#8A93A0":"#E3E6EC"},s);el("text",{x:m.l-8,y:Y(v)+4,"text-anchor":"end","font-size":12},s,v+" %")}
 years.forEach(y=>el("text",{x:X(y),y:H-10,"text-anchor":"middle","font-size":12,"font-weight":y===x1?700:400},s,y));
 [["contract","#0B1F40"],["effective","#8B7543"]].forEach(([k,c])=>{
  if(pts.length){el("polyline",{points:pts.map(p=>`${X(p.year)},${Y(p[k])}`).join(" "),fill:"none",stroke:c,"stroke-width":3},s);
   const last=pts[pts.length-1];el("line",{x1:X(last.year),y1:Y(last[k]),x2:X(x1),y2:Y(fc[k]),stroke:c,"stroke-width":2.5,"stroke-dasharray":"6 5"},s)}
  pts.forEach(p=>el("circle",{cx:X(p.year),cy:Y(p[k]),r:4,fill:c},s));
  el("circle",{cx:X(x1),cy:Y(fc[k]),r:6,fill:"#fff",stroke:c,"stroke-width":3},s);
  el("text",{x:X(x1)+10,y:Y(fc[k])+4,"font-size":13,"font-weight":700,fill:c},s,fr(fc[k])+" %")});
 if(pts.length&&pts[0].year>x0)el("text",{x:X(x0),y:m.t+12,"font-size":12,fill:"#8A93A0"},s,`Historique disponible à partir de ${pts[0].year} (moins de ${D.min_pairs} paires avant)`)}
function mix(){const box=document.getElementById("mix");box.innerHTML="";const rows=cur.mix,W=460,rh=46,m={l:92,r:60,t:6},H=m.t+rows.length*rh+8;
 const mx=Math.max(...rows.map(r=>r.renewal+r.new)),X=v=>v/mx*(W-m.l-m.r);const s=svg(W,H);box.appendChild(s);
 rows.forEach((r,i)=>{const y=m.t+i*rh;el("text",{x:m.l-10,y:y+22,"text-anchor":"end","font-size":13},s,r.label);
  el("rect",{x:m.l,y:y+8,width:X(r.renewal),height:22,fill:"#0B1F40",rx:2},s);el("rect",{x:m.l+X(r.renewal),y:y+8,width:X(r.new),height:22,fill:"#B9AB88",rx:2},s);
  el("text",{x:m.l+X(r.renewal+r.new)+8,y:y+24,"font-size":13,"font-weight":700},s,Math.round(r.renewal+r.new))});
 const segs=D.segments[scen],p=cur.province==="Ontario"?["Ontario"]:cur.province==="Québec"?["Québec"]:["Québec","Ontario"];
 const t=document.createElement("p");t.className="note";t.style.marginTop="8px";
 t.innerHTML=p.map(pr=>`<b>${pr}</b> — renouvellement ${fr(segs[pr+"|renouvellement"])} % encaissé, nouveau locataire ${fr(segs[pr+"|nouveau locataire"])} %`).join("<br>");box.appendChild(t)}
function cmp(){const box=document.getElementById("cmp");box.innerHTML="";const es=[...D.entries].sort((a,b)=>eff(b)-eff(a));
 const W=1100,rh=34,m={l:150,r:70,t:6},H=m.t+es.length*rh+24,mx=Math.max(...es.map(e=>Math.max(e.contract,eff(e))))*1.05,X=v=>v/mx*(W-m.l-m.r);const s=svg(W,H);box.appendChild(s);
 es.forEach((e,i)=>{const y=m.t+i*rh,on=e===cur,g=el("g",{style:"cursor:pointer"},s);g.onclick=()=>{cur=e;render()};
  el("text",{x:m.l-10,y:y+20,"text-anchor":"end","font-size":14,"font-weight":on?700:400},g,e.name);
  el("rect",{x:m.l,y:y+6,width:X(e.contract),height:20,fill:"#E3E6EC",rx:2},g);
  el("rect",{x:m.l,y:y+6,width:X(eff(e)),height:20,fill:on?"#8B7543":"#B9AB88",rx:2},g);
  el("text",{x:m.l+X(Math.max(e.contract,eff(e)))+8,y:y+21,"font-size":13,"font-weight":on?700:400},g,fr(eff(e))+" %")});
 el("line",{x1:m.l+X(eff(PF)),x2:m.l+X(eff(PF)),y1:0,y2:H-18,stroke:"#0B1F40","stroke-dasharray":"4 4"},s);
 el("text",{x:m.l+X(eff(PF)),y:H-4,"text-anchor":"middle","font-size":12},s,"portefeuille "+fr(eff(PF))+" %");
 el("text",{x:W-m.r,y:H-4,"text-anchor":"end","font-size":12,fill:"#8A93A0"},s,"barre claire : hausse au bail")}
function render(){
 buttons("tabs",[...D.entries].sort((a,b)=>a===PF?-1:b===PF?1:a.name.localeCompare(b.name)).map(e=>({label:e.name,e})),it=>it.e===cur,it=>{cur=it.e});
 buttons("scens",D.scenarios,it=>it.id===scen,it=>{scen=it.id});
 document.getElementById("scen-note").textContent="Scénario "+D.scenarios.find(s=>s.id===scen).label.toLowerCase()+" : "+D.scenarios.find(s=>s.id===scen).note+".";
 kpis();hist();mix();cmp();
 document.getElementById("note").innerHTML=`Les taux sont estimés par segment (Québec / Ontario × renouvellement / nouveau locataire) : un immeuble ne diffère que par son régime et sa part de renouvellements. Le scénario ne change que la hausse des concessions, donc la hausse encaissée. Historique : médiane des paires même logement (au moins ${D.min_pairs} paires par point). Baux attendus : baux en cours qui finissent d'ici fin ${D.year} dans l'extrait (pondération seulement). Aucune donnée individuelle du CRM.`}
render();
</script></body></html>"""
DASHBOARD_FILE.write_text(DASHBOARD_TEMPLATE.replace("__DATA__", json.dumps(dashboard_data, ensure_ascii=False)), encoding="utf-8")
print(f"{DASHBOARD_FILE} : {len(entries) - 1} immeubles + portefeuille, {len(DASHBOARD_SCENARIOS)} scénarios")

## Références

**Données publiques** (téléchargement et nettoyage : `public_data.py`, section 2.1)
- Statistique Canada, Tableau 18-10-0004-01, Indice des prix à la consommation (loyers, logement locatif, ensemble). https://www150.statcan.gc.ca/t1/tbl1/fr/tv.action?pid=1810000401
- Tribunal administratif du logement, communiqués annuels de fixation de loyer (2016, 2022–2025). https://www.tal.gouv.qc.ca
- Taux TAL 2026 (3,1 %) : Radio-Canada, 2026. https://ici.radio-canada.ca/nouvelle/2221781/loyers-tal-logement-augmentation-locataires — nouveau règlement : Morneau Sénéchal avocats, 29 déc. 2025.
- Gouvernement de l'Ontario, *Rent increase guideline* (lignes directrices 1991–2027 et exemption post-15 nov. 2018). https://www.ontario.ca/page/rent-increase-guideline
- SCHL, Enquête sur les logements locatifs, tableaux de données 2021–2025 (Canada, Montréal, Ottawa). https://www.cmhc-schl.gc.ca
- FPI résidentielles : InterRent (rapports de gestion T4 2019–2024), Killam (T4 2023, 2025), CAPREIT (communiqués, rapport annuel 2023, présentation T4 2025), Minto (communiqués T4 2023–2025). Liens dans `external/clean/reit_metrics.csv`.
- Kaggle, *25000+ Canadian rental housing market (June 2024)*, S. Gavrylov. https://www.kaggle.com/datasets/sergiygavrylov/25000-canadian-rental-housing-market-june-2024

**Méthodes**
- Indice de ventes répétées (repeat-sales) : Bailey, Muth et Nourse (1963), *A Regression Method for Real Estate Price Index Construction*, JASA ; Case et Shiller (1987).
- Indices hédoniques : Rosen (1974), *Hedonic Prices and Implicit Markets*, JPE.
- Valeurs SHAP : Lundberg et Lee (2017), *A Unified Approach to Interpreting Model Predictions*, NeurIPS.
- Évaluation par origine glissante : Tashman (2000), *Out-of-sample tests of forecasting accuracy*, IJF ; Hyndman et Athanasopoulos, *Forecasting: Principles and Practice* (3e éd.).

**Outils d'IA**
- Claude (Anthropic), modèle `claude-opus-5-5`, utilisé pour rechercher et télécharger les sources publiques, écrire le code de préparation des données et d'analyse, et rédiger la documentation. Les décisions de modélisation, analyses, datasets publics, hypothèses ont été choisis et validés par l'équipe.